In [1]:
# ============================================================
# CELL 0 — COLAB SETUP
# Run this FIRST at the start of every new Colab session.
# Takes ~1 min. Safe to re-run.
# ============================================================

# 1. Mount Google Drive
import os
import shutil
from google.colab import drive

mountpoint = '/content/drive'

# Unmount and clean up any stale mount
try:
    drive.flush_and_unmount()
except Exception:
    pass  # Ignore if not mounted yet

# Remove stale directory so mount() gets a clean slate
if os.path.exists(mountpoint):
    shutil.rmtree(mountpoint)

drive.mount(mountpoint)

# 2. Install packages not pre-installed on Colab
!pip install praat-parselmouth noisereduce soundfile transformers torchaudio -q

# 3. Create all project directories on Drive (idempotent)
PROJECT_ROOT = "/content/drive/MyDrive/team_databaes"

dirs = [
    "outputs",
    "data/train/native",
    "data/train/non_native",
    "data/preprocessed/native",
    "data/preprocessed/non_native",
    "data/embeddings",
    "data/segments/native",
    "data/segments/non_native",
    "data/augmented/native",
    "data/augmented/non_native",
]
for d in dirs:
    os.makedirs(f"{PROJECT_ROOT}/{d}", exist_ok=True)

# 4. Verify GPU (critical for Stage 6A)
import torch
print(f"\nGPU available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU           : {torch.cuda.get_device_name(0)}")
    print(f"VRAM          : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU — Runtime → Change runtime type → T4 GPU")
    print("   Stage 6A will be very slow on CPU.")

# 5. Check renan_dataset.csv is on Drive
csv_path = f"{PROJECT_ROOT}/renan_dataset.csv"
if os.path.exists(csv_path):
    import pandas as pd
    df = pd.read_csv(csv_path)
    print(f"\n✅ renan_dataset.csv found ({len(df)} rows)")
else:
    print(f"\n❌ renan_dataset.csv NOT found.")
    print(f"   Upload it manually to: {PROJECT_ROOT}/")
    print(f"   (Google Drive → team_databaes folder → Upload file)")

print(f"\nProject root  : {PROJECT_ROOT}")
print(f"Setup complete ✅")

Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 74.3 MB/s eta 0:00:00

GPU available : True
GPU           : Tesla T4
VRAM          : 15.6 GB

✅ renan_dataset.csv found (178 rows)

Project root  : /content/drive/MyDrive/team_databaes
Setup complete ✅


In [2]:
# ============================================================
# CELL 1 — STAGE 1: DATA EXPLORATION
# Reads renan_dataset.csv from Drive.
# All outputs saved directly to Drive.
# Safe to re-run — overwrites previous outputs.
# ============================================================

import os, io, time, requests, librosa, numpy as np, pandas as pd
from tqdm.notebook import tqdm

PROJECT_ROOT = "/content/drive/MyDrive/team_databaes"
EXCEL_PATH   = f"{PROJECT_ROOT}/renan_dataset.csv"
OUTPUT_DIR   = f"{PROJECT_ROOT}/outputs"
TARGET_SR    = 16000

# Set to None to explore all 160 files (takes ~5 min)
# Keep at 50 for a quick sanity check (~1 min)
EXPLORE_N = None

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Load spreadsheet ──────────────────────────────────────────
print("📂 Loading dataset spreadsheet...")
df_raw = pd.read_csv(EXCEL_PATH)
print(f"   Total rows loaded: {len(df_raw)}")
print(f"   Columns: {list(df_raw.columns)}")

df_raw["label"] = df_raw["nativity_status"].map({"Native": 1, "Non-Native": 0})

unknown_labels = df_raw[df_raw["label"].isna()]
if len(unknown_labels) > 0:
    print(f"⚠️  {len(unknown_labels)} rows with unrecognised nativity_status:")
    print(unknown_labels["nativity_status"].unique())

df_explore = df_raw.sample(n=EXPLORE_N, random_state=42) if EXPLORE_N else df_raw
print(f"\n🔍 Exploring {len(df_explore)} files...\n")

# ── Download + inspect ────────────────────────────────────────
def download_and_inspect(row):
    dp_id   = row["dp_id"]
    url     = row["audio_url"]
    label   = row["label"]
    dialect = row["language"]

    result = {
        "dp_id":    dp_id, "label": label,
        "nativity": row["nativity_status"], "dialect": dialect, "url": url,
    }
    try:
        response    = requests.get(url, timeout=15)
        response.raise_for_status()
        audio_bytes = io.BytesIO(response.content)

        y, sr = librosa.load(audio_bytes, sr=None, mono=True)
        duration      = librosa.get_duration(y=y, sr=sr)
        rms           = float(np.sqrt(np.mean(y ** 2)))
        zcr           = float(np.mean(librosa.feature.zero_crossing_rate(y)))
        frame_rms     = librosa.feature.rms(y=y)[0]
        silence_ratio = float(np.mean(frame_rms < 0.01 * frame_rms.max()))

        result.update({
            "original_sr":    sr,
            "duration_s":     round(duration, 2),
            "rms_volume":     round(rms, 5),
            "zero_cross_rate":round(zcr, 5),
            "silence_ratio":  round(silence_ratio, 3),
            "n_samples":      len(y),
            "usable":         duration >= 3.0,
            "error":          None,
        })
    except requests.exceptions.Timeout:
        result.update({"error": "TIMEOUT", "usable": False})
    except requests.exceptions.HTTPError as e:
        result.update({"error": f"HTTP_{e.response.status_code}", "usable": False})
    except Exception as e:
        result.update({"error": str(e)[:80], "usable": False})
    return result

records = []
for _, row in tqdm(df_explore.iterrows(), total=len(df_explore), desc="Inspecting audio"):
    records.append(download_and_inspect(row))
    time.sleep(0.05)

df_stats = pd.DataFrame(records)

# ── Print report ──────────────────────────────────────────────
sep = "=" * 58
report_lines = []

def log(line=""):
    print(line)
    report_lines.append(line)

log(sep)
log("  STAGE 1 — DATA EXPLORATION REPORT")
log(f"  Dataset: renan_dataset.csv  |  Files inspected: {len(df_stats)}")
log(sep)

log("\n📊 CLASS BALANCE")
log("-" * 35)
balance = df_raw["nativity_status"].value_counts()
for name, count in balance.items():
    pct = 100 * count / len(df_raw)
    log(f"   {name:<15}: {count:>5} files  ({pct:.1f}%)")
imbalance_ratio = balance.max() / balance.min()
log(f"\n   Imbalance ratio: {imbalance_ratio:.1f}x")
if imbalance_ratio > 2:
    log("   ⚠️  Significant imbalance — class_weight='balanced' is essential in Stage 8")

log("\n🌍 DIALECT DISTRIBUTION")
log("-" * 35)
for dialect, count in df_raw["language"].value_counts().items():
    pct = 100 * count / len(df_raw)
    log(f"   {dialect:<18}: {count:>4} files  ({pct:.1f}%)")

log("\n⏱️  DURATION STATS (seconds)")
log("-" * 35)
valid = df_stats[df_stats["error"].isna()]
for label_val, label_name in [(1, "Native"), (0, "Non-Native")]:
    subset = valid[valid["label"] == label_val]["duration_s"]
    if len(subset) == 0:
        continue
    log(f"\n   {label_name}:")
    log(f"     mean={subset.mean():.1f}s  median={subset.median():.1f}s  "
        f"min={subset.min():.1f}s  max={subset.max():.1f}s")
    est_segs = subset.apply(lambda d: max(0, int((d - 3) / 1) + 1))
    log(f"     Estimated 3s segments per file: ~{est_segs.mean():.0f} avg, "
        f"{est_segs.sum()} total")

log("\n🎵 SAMPLE RATES FOUND (original, before resampling)")
log("-" * 35)
for sr, count in valid["original_sr"].value_counts().items():
    flag = "✅" if sr == TARGET_SR else "⚠️  will resample"
    log(f"   {sr} Hz : {count} files  {flag}")

log("\n🔊 VOLUME (RMS) BY CLASS")
log("-" * 35)
rms_stats = valid.groupby("label")["rms_volume"].describe()
log(rms_stats.round(5).to_string())
rms_native    = valid[valid["label"]==1]["rms_volume"].mean()
rms_nonnative = valid[valid["label"]==0]["rms_volume"].mean()
if abs(rms_native - rms_nonnative) / max(rms_native, rms_nonnative) > 0.2:
    log("   ⚠️  Noticeable volume difference — RMS normalization in Stage 3 is important")

log("\n🤫 SILENCE RATIO BY CLASS  (0=no silence, 1=all silence)")
log("-" * 35)
log(valid.groupby("label")["silence_ratio"].describe().round(3).to_string())

log("\n⚠️  PROBLEMATIC FILES")
log("-" * 35)
errors = df_stats[df_stats["error"].notna()]
short  = df_stats[(df_stats["error"].isna()) & (df_stats["duration_s"] < 3.0)]
log(f"   Download/load errors : {len(errors)}")
log(f"   Too short (<3s)      : {len(short)}  ← will be dropped in Stage 3")
log(f"   Usable files         : {df_stats['usable'].sum()} / {len(df_stats)}")
if len(errors) > 0:
    log("\n   Error breakdown:")
    log(errors["error"].value_counts().to_string())

log("\n" + sep)
log("  KEY TAKEAWAYS FOR DOWNSTREAM STAGES")
log(sep)
log(f"  Stage 3: Resample all audio to {TARGET_SR}Hz mono + RMS normalize")
log(f"  Stage 4: Use 3s window / 1s hop for segmentation")
log(f"  Stage 5: Augment training data only (non-native is minority class)")
log(f"  Stage 8: Use class_weight='balanced' in SVM due to imbalance")
log(sep)

# ── Save outputs directly to Drive ───────────────────────────
df_stats.to_csv(f"{OUTPUT_DIR}/stage1_exploration.csv", index=False)
log(f"\n💾 Saved: {OUTPUT_DIR}/stage1_exploration.csv")

with open(f"{OUTPUT_DIR}/stage1_report.txt", "w") as f:
    f.write("\n".join(report_lines))
log(f"💾 Saved: {OUTPUT_DIR}/stage1_report.txt")

print("\n✅ Stage 1 complete.")

📂 Loading dataset spreadsheet...
   Total rows loaded: 178
   Columns: ['dp_id', 'audio_url', 'nativity_status', 'language', 'audio_path']

🔍 Exploring 178 files...



Inspecting audio:   0%|          | 0/178 [00:00<?, ?it/s]

  STAGE 1 — DATA EXPLORATION REPORT
  Dataset: renan_dataset.csv  |  Files inspected: 178

📊 CLASS BALANCE
-----------------------------------
   Native         :   114 files  (64.0%)
   Non-Native     :    64 files  (36.0%)

   Imbalance ratio: 1.8x

🌍 DIALECT DISTRIBUTION
-----------------------------------
   Arabic_SA         :   63 files  (35.4%)
   Arabic_QA         :   56 files  (31.5%)
   Arabic_AE         :   38 files  (21.3%)
   Arabic_KW         :    3 files  (1.7%)

⏱️  DURATION STATS (seconds)
-----------------------------------

   Native:
     mean=45.4s  median=42.1s  min=17.2s  max=118.3s
     Estimated 3s segments per file: ~43 avg, 4640 total

   Non-Native:
     mean=54.0s  median=36.7s  min=25.6s  max=400.2s
     Estimated 3s segments per file: ~51 avg, 2214 total

🎵 SAMPLE RATES FOUND (original, before resampling)
-----------------------------------
   16000.0 Hz : 107 files  ✅
   48000.0 Hz : 35 files  ⚠️  will resample
   44100.0 Hz : 8 files  ⚠️  will resample


In [3]:
# ============================================================
# RECOVERY CELL — Fix all 27 failed files from Stage 1
# Run this AFTER Cell 1 (Stage 1) and BEFORE Cell 3 (Stage 2)
#
# Fixes:
#   1. Google Drive sharing links (rows 160–177) → direct URLs
#   2. AAC format files (rows 91, 98) → install ffmpeg
#   3. CloudFront WAV/MP3 failures → User-Agent header + retry
#
# After this cell runs successfully, re-run Cell 1 (Stage 1)
# to verify all files now load, then proceed to Stage 2.
# ============================================================

import os, io, re, time, requests
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from tqdm.notebook import tqdm

# ── Step 0: Install ffmpeg (fixes AAC support in librosa) ────
print("Step 0: Installing ffmpeg for AAC support...")
os.system("apt-get install -y ffmpeg -q")
print("✅ ffmpeg installed\n")

PROJECT_ROOT = "/content/drive/MyDrive/team_databaes"
CSV_PATH     = f"{PROJECT_ROOT}/renan_dataset.csv"
TIMEOUT      = 20

# ── Load CSV ─────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH)
print(f"Loaded CSV: {len(df)} rows\n")

# ════════════════════════════════════════════════════════════
# FIX 1 — Assign missing dp_ids and dialects (rows 160–177)
# These are all Non-Native files added via Google Drive links
# ════════════════════════════════════════════════════════════
print("=" * 55)
print("FIX 1 — Assigning missing dp_ids and dialects")
print("=" * 55)

missing_mask = df["dp_id"].isna()
print(f"Rows with missing dp_id: {missing_mask.sum()}")

if missing_mask.sum() > 0:
    # Generate new unique dp_ids starting from max existing + 1
    max_id = df["dp_id"].dropna().max()
    new_ids = range(int(max_id) + 1, int(max_id) + 1 + missing_mask.sum())
    df.loc[missing_mask, "dp_id"] = list(new_ids)

    # These are all Non-Native — assign dialect as "Unknown_NN"
    # Update this manually if you know the actual dialects
    df.loc[missing_mask, "language"] = df.loc[missing_mask, "language"].fillna("Unknown_NN")

    print(f"Assigned dp_ids: {list(new_ids)}")
    print(f"Assigned dialect: Unknown_NN (update manually if you know their dialects)")

# ════════════════════════════════════════════════════════════
# FIX 2 — Convert Google Drive sharing links to direct URLs
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 55)
print("FIX 2 — Converting Google Drive sharing links")
print("=" * 55)

def fix_gdrive_url(url):
    """Convert /view?usp=sharing → direct download URL."""
    if not isinstance(url, str):
        return url
    match = re.search(r'/file/d/([a-zA-Z0-9_-]+)', url)
    if match:
        file_id = match.group(1)
        return f"https://drive.google.com/uc?export=download&id={file_id}"
    return url

gdrive_mask = df["audio_url"].str.contains("drive.google.com", na=False)
print(f"Google Drive URLs found: {gdrive_mask.sum()}")

df.loc[gdrive_mask, "audio_url"] = df.loc[gdrive_mask, "audio_url"].apply(fix_gdrive_url)

# Verify conversion
converted = df.loc[gdrive_mask, "audio_url"].str.contains("uc?export=download", na=False).sum()
print(f"Successfully converted: {converted} / {gdrive_mask.sum()}")
print("Sample converted URL:", df.loc[gdrive_mask, "audio_url"].iloc[0])

# ════════════════════════════════════════════════════════════
# FIX 3 — Save updated CSV back to Drive
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 55)
print("FIX 3 — Saving updated CSV")
print("=" * 55)

# Backup original first
backup_path = CSV_PATH.replace(".csv", "_backup.csv")
if not os.path.exists(backup_path):
    import shutil
    shutil.copy(CSV_PATH, backup_path)
    print(f"Original CSV backed up to: {backup_path}")

df.to_csv(CSV_PATH, index=False)
print(f"Updated CSV saved to: {CSV_PATH}")

# ════════════════════════════════════════════════════════════
# VERIFY — Test-load all previously failed files
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 55)
print("VERIFY — Test loading all 27 previously failed files")
print("=" * 55)

# These are the row indices that failed in Stage 1
FAILED_ROWS = [16, 27, 43, 91, 98, 104, 131, 134, 139,
               160, 161, 162, 163, 164, 165, 166, 167, 168,
               169, 170, 171, 172, 173, 174, 175, 176, 177]

HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

results = []

for idx in tqdm(FAILED_ROWS, desc="Testing recovery"):
    row  = df.iloc[idx]
    url  = row["audio_url"]
    dpid = row["dp_id"]

    result = {"row": idx, "dp_id": dpid, "url": url,
              "nativity": row["nativity_status"],
              "dialect": row["language"],
              "status": None, "duration_s": None, "error": None}
    try:
        resp = requests.get(url, timeout=TIMEOUT, headers=HEADERS)
        resp.raise_for_status()

        # Check content type — HTML means the URL is still wrong
        ctype = resp.headers.get("Content-Type", "")
        if "text/html" in ctype:
            result["status"] = "FAILED"
            result["error"]  = f"Got HTML page instead of audio (Content-Type: {ctype})"
            results.append(result)
            continue

        audio_bytes = io.BytesIO(resp.content)
        y, sr       = librosa.load(audio_bytes, sr=None, mono=True)
        duration    = librosa.get_duration(y=y, sr=sr)

        result["status"]     = "✅ OK"
        result["duration_s"] = round(duration, 1)

    except Exception as e:
        result["status"] = "FAILED"
        result["error"]  = str(e)[:100]

    results.append(result)
    time.sleep(0.1)

# ── Summary ───────────────────────────────────────────────────
res_df = pd.DataFrame(results)
ok     = res_df[res_df["status"] == "✅ OK"]
failed = res_df[res_df["status"] == "FAILED"]

print(f"\n{'=' * 55}")
print(f"  RECOVERY RESULTS")
print(f"{'=' * 55}")
print(f"  ✅ Recovered : {len(ok)} / {len(FAILED_ROWS)} files")
print(f"  ❌ Still failing: {len(failed)} files")

if len(ok) > 0:
    print(f"\n  Recovered files:")
    print(ok[["row", "dp_id", "nativity", "dialect", "duration_s"]].to_string(index=False))

if len(failed) > 0:
    print(f"\n  Still failing (manual fix needed):")
    print(failed[["row", "dp_id", "nativity", "url", "error"]].to_string(index=False))
    print("\n  ⚠️  For files still showing HTML errors:")
    print("     → The Google Drive files may not be publicly shared.")
    print("     → Ask the file owner to set sharing to 'Anyone with the link'.")
    print("     → For CloudFront files, try opening the URL manually in a browser.")

print(f"\n{'=' * 55}")
print(f"  NEXT STEPS")
print(f"{'=' * 55}")
print(f"  1. If any GDrive files still fail → make them public (see above)")
print(f"  2. Re-run Cell 1 (Stage 1) to confirm all files now load")
print(f"  3. Proceed to Cell 3 (Stage 2) — the updated CSV will be used")
print(f"{'=' * 55}")

Step 0: Installing ffmpeg for AAC support...
✅ ffmpeg installed

Loaded CSV: 178 rows

FIX 1 — Assigning missing dp_ids and dialects
Rows with missing dp_id: 18
Assigned dp_ids: [794, 795, 796, 797, 798, 799, 800, 801, 802, 803, 804, 805, 806, 807, 808, 809, 810, 811]
Assigned dialect: Unknown_NN (update manually if you know their dialects)

FIX 2 — Converting Google Drive sharing links
Google Drive URLs found: 18
Successfully converted: 0 / 18
Sample converted URL: https://drive.google.com/uc?export=download&id=1CiJJuVj2gU2A_GmnNuxXCHPu0hf-WhuK

FIX 3 — Saving updated CSV
Original CSV backed up to: /content/drive/MyDrive/team_databaes/renan_dataset_backup.csv
Updated CSV saved to: /content/drive/MyDrive/team_databaes/renan_dataset.csv

VERIFY — Test loading all 27 previously failed files


Testing recovery:   0%|          | 0/27 [00:00<?, ?it/s]


  RECOVERY RESULTS
  ✅ Recovered : 18 / 27 files
  ❌ Still failing: 9 files

  Recovered files:
 row  dp_id   nativity    dialect  duration_s
 160  794.0 Non-Native Unknown_NN        51.0
 161  795.0 Non-Native Unknown_NN       219.0
 162  796.0 Non-Native Unknown_NN       130.0
 163  797.0 Non-Native Unknown_NN        25.0
 164  798.0 Non-Native Unknown_NN        71.0
 165  799.0 Non-Native Unknown_NN        27.0
 166  800.0 Non-Native Unknown_NN       105.0
 167  801.0 Non-Native Unknown_NN        26.0
 168  802.0 Non-Native Unknown_NN        61.0
 169  803.0 Non-Native Unknown_NN        32.0
 170  804.0 Non-Native Unknown_NN       125.0
 171  805.0 Non-Native Unknown_NN        20.0
 172  806.0 Non-Native Unknown_NN        19.0
 173  807.0 Non-Native Unknown_NN        27.0
 174  808.0 Non-Native Unknown_NN        20.0
 175  809.0 Non-Native Unknown_NN        27.0
 176  810.0 Non-Native Unknown_NN        53.0
 177  811.0 Non-Native Unknown_NN       103.0

  Still failing (manual fix 

In [4]:
# ============================================================
# RECOVERY CELL 2 — Fix remaining 9 CloudFront files
#
# Root cause: librosa can't use ffmpeg on a BytesIO object.
# Fix: download to a TEMP FILE on disk, then load from path.
# ffmpeg can then handle WAV/MP3/AAC properly.
# ============================================================

import os, io, time, tempfile, subprocess, requests
import numpy as np
import pandas as pd
import librosa
from tqdm.notebook import tqdm

PROJECT_ROOT = "/content/drive/MyDrive/team_databaes"
CSV_PATH     = f"{PROJECT_ROOT}/renan_dataset.csv"
TIMEOUT      = 30
HEADERS      = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

# Rows that are still failing
STILL_FAILING = [16, 27, 43, 91, 98, 104, 131, 134, 139]

df = pd.read_csv(CSV_PATH)

print("=" * 55)
print("  RECOVERY CELL 2 — CloudFront files (disk-based load)")
print("=" * 55)

def download_and_load(url, timeout=TIMEOUT):
    """
    Download audio to a temp file on disk, then load with librosa.
    Falls back to ffmpeg conversion if direct load fails.
    Returns (y, sr, duration, error_message)
    """
    # Get file extension from URL
    ext = os.path.splitext(url.split("?")[0])[-1].lower()
    if ext not in [".wav", ".mp3", ".aac", ".ogg", ".flac", ".m4a"]:
        ext = ".mp3"  # default fallback

    try:
        resp = requests.get(url, timeout=timeout, headers=HEADERS)
        resp.raise_for_status()

        # Check we actually got audio not HTML
        ctype = resp.headers.get("Content-Type", "")
        if "text/html" in ctype:
            return None, None, None, f"Got HTML page (URL may be dead or auth-required)"

        # Write to temp file
        with tempfile.NamedTemporaryFile(suffix=ext, delete=False) as tmp:
            tmp.write(resp.content)
            tmp_path = tmp.name

        # Try loading directly first
        try:
            y, sr = librosa.load(tmp_path, sr=None, mono=True)
            duration = librosa.get_duration(y=y, sr=sr)
            os.unlink(tmp_path)
            return y, sr, duration, None
        except Exception as e1:
            # Fallback: use ffmpeg to convert to WAV first
            wav_path = tmp_path.replace(ext, "_converted.wav")
            try:
                subprocess.run(
                    ["ffmpeg", "-y", "-i", tmp_path, "-ar", "16000",
                     "-ac", "1", wav_path],
                    capture_output=True, check=True
                )
                y, sr = librosa.load(wav_path, sr=None, mono=True)
                duration = librosa.get_duration(y=y, sr=sr)
                os.unlink(tmp_path)
                os.unlink(wav_path)
                return y, sr, duration, None
            except subprocess.CalledProcessError as e2:
                os.unlink(tmp_path)
                if os.path.exists(wav_path):
                    os.unlink(wav_path)
                return None, None, None, f"ffmpeg conversion failed: {str(e2)[:80]}"
            except Exception as e3:
                return None, None, None, f"Load after ffmpeg failed: {str(e3)[:80]}"

    except requests.exceptions.HTTPError as e:
        return None, None, None, f"HTTP {e.response.status_code}"
    except requests.exceptions.Timeout:
        return None, None, None, "TIMEOUT"
    except Exception as e:
        return None, None, None, str(e)[:100]


# ── Test all 9 remaining files ────────────────────────────────
results = []
for idx in tqdm(STILL_FAILING, desc="Testing CloudFront files"):
    row = df.iloc[idx]
    url = row["audio_url"]

    y, sr, duration, error = download_and_load(url)

    results.append({
        "row":      idx,
        "dp_id":    row["dp_id"],
        "nativity": row["nativity_status"],
        "dialect":  row["language"],
        "url":      url,
        "status":   "✅ OK" if error is None else "❌ FAILED",
        "duration_s": duration,
        "sr":         sr,
        "error":      error,
    })
    time.sleep(0.2)

# ── Results ───────────────────────────────────────────────────
res_df = pd.DataFrame(results)
ok     = res_df[res_df["status"] == "✅ OK"]
failed = res_df[res_df["status"] == "❌ FAILED"]

print(f"\n{'=' * 55}")
print(f"  RESULTS")
print(f"{'=' * 55}")
print(f"  ✅ Recovered : {len(ok)} / {len(STILL_FAILING)}")
print(f"  ❌ Still failing: {len(failed)}")

if len(ok) > 0:
    print(f"\n  Recovered:")
    print(ok[["row", "dp_id", "nativity", "dialect", "duration_s", "sr"]].to_string(index=False))

if len(failed) > 0:
    print(f"\n  Still failing:")
    print(failed[["row", "dp_id", "nativity", "url", "error"]].to_string(index=False))

    # ── If still failing: check if URLs are even alive ────────
    print(f"\n  Checking if dead URLs return audio at all...")
    for _, r in failed.iterrows():
        try:
            head = requests.head(r["url"], timeout=10, headers=HEADERS)
            ctype = head.headers.get("Content-Type", "unknown")
            print(f"  Row {r['row']}: HTTP {head.status_code} | Content-Type: {ctype}")
        except Exception as e:
            print(f"  Row {r['row']}: HEAD request failed — {e}")

    print(f"""
  ── What to do with files that still fail ──────────────
  Option A (preferred): Ask Renan to re-upload those files.
                        They appear to be corrupted on the server.

  Option B: Drop them from the dataset.
            These are {len(failed[failed['nativity']=='Native'])} Native and
            {len(failed[failed['nativity']=='Non-Native'])} Non-Native files —
            dropping them has minimal impact given your dataset size.

  To drop them permanently, run the cell below.
  ────────────────────────────────────────────────────────
    """)

# ── Also update Stage 1's load_audio to use disk-based loading ──
print("=" * 55)
print("  IMPORTANT — Update Stage 2 download function")
print("=" * 55)
print("""
  The Stage 2 download function also loads from BytesIO.
  Add this to the top of Cell 3 (Stage 2) to make all
  downloads use disk-based loading automatically:

      import tempfile, subprocess

  And replace the librosa.load(audio_bytes, ...) line with:

      with tempfile.NamedTemporaryFile(suffix='.mp3', delete=False) as tmp:
          tmp.write(response.content)
          tmp_path = tmp.name
      try:
          y, sr = librosa.load(tmp_path, sr=None, mono=True)
      finally:
          os.unlink(tmp_path)
""")


  RECOVERY CELL 2 — CloudFront files (disk-based load)


Testing CloudFront files:   0%|          | 0/9 [00:00<?, ?it/s]

/tmp/ipython-input-835/2216870237.py:56: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(tmp_path, sr=None, mono=True)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipython-input-835/2216870237.py:56: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(tmp_path, sr=None, mono=True)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipython-input-835/2216870237.py:56: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(tmp_path, sr=None, mono=True)
/us


  RESULTS
  ✅ Recovered : 9 / 9
  ❌ Still failing: 0

  Recovered:
 row  dp_id   nativity   dialect  duration_s    sr
  16  529.0     Native Arabic_QA   39.552000 16000
  27  656.0     Native Arabic_SA  112.320000 16000
  43  578.0 Non-Native Arabic_SA   35.136000 16000
  91  332.0     Native Arabic_AE   34.346667 48000
  98  351.0     Native Arabic_QA   34.752000 16000
 104  355.0     Native Arabic_SA   46.420000 48000
 131  532.0 Non-Native Arabic_QA   39.872000 16000
 134  479.0     Native Arabic_SA   46.420000 48000
 139  407.0 Non-Native Arabic_QA   34.112000 16000
  IMPORTANT — Update Stage 2 download function

  The Stage 2 download function also loads from BytesIO.
  Add this to the top of Cell 3 (Stage 2) to make all
  downloads use disk-based loading automatically:

      import tempfile, subprocess

  And replace the librosa.load(audio_bytes, ...) line with:

      with tempfile.NamedTemporaryFile(suffix='.mp3', delete=False) as tmp:
          tmp.write(response.content)
  

In [5]:
# ============================================================
# CELL 3 — STAGE 2: DATA COLLECTION (UPDATED)
# Changes from original:
#   - Uses disk-based loading (temp file) instead of BytesIO
#     so ffmpeg can decode WAV/MP3/AAC/any format correctly
#   - Applies User-Agent header to all requests
#   - Auto-detects file extension from URL for temp file
# ============================================================

import os, io, time, tarfile, hashlib, tempfile, subprocess, requests
import numpy as np, pandas as pd
from tqdm.notebook import tqdm

PROJECT_ROOT    = "/content/drive/MyDrive/team_databaes"
RENAN_CSV       = f"{PROJECT_ROOT}/renan_dataset.csv"
TRAIN_NATIVE    = f"{PROJECT_ROOT}/data/train/native"
TRAIN_NONNATIVE = f"{PROJECT_ROOT}/data/train/non_native"
OUTPUT_DIR      = f"{PROJECT_ROOT}/outputs"

# ── ⚠️  PASTE YOUR API KEY HERE ──────────────────────────────
CV_API_KEY = "0a1770b3c333e1cc9175c75a74337695384ea6d42931e26a4adfab8bc05847c0"
# ─────────────────────────────────────────────────────────────

CV_DATASET_ID  = "cmj8u3os6000tnxxb169x1zdc"
CV_API_URL     = f"https://datacollective.mozillafoundation.org/api/datasets/{CV_DATASET_ID}/download"
CV_NATIVE_N    = 200
CV_MIN_UPVOTES = 2
TIMEOUT        = 30
SLEEP          = 0.1
HEADERS        = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

# ════════════════════════════════════════════════════════════
# PART A — DOWNLOAD RENAN FILES
# ════════════════════════════════════════════════════════════
print("=" * 58)
print("  PART A — Downloading Renan Training Audio (178 files)")
print("=" * 58)

df_renan = pd.read_csv(RENAN_CSV)
df_renan["label"]     = df_renan["nativity_status"].map({"Native": 1, "Non-Native": 0})
df_renan["filename"]  = df_renan["dp_id"].apply(lambda x: f"dp_{int(x)}.mp3")
df_renan["save_dir"]  = df_renan["label"].map({1: TRAIN_NATIVE, 0: TRAIN_NONNATIVE})
df_renan["save_path"] = df_renan.apply(
    lambda r: os.path.join(r["save_dir"], r["filename"]), axis=1
)

manifest_records = []

def get_ext_from_url(url):
    """Extract file extension from URL, default to .mp3."""
    ext = os.path.splitext(url.split("?")[0])[-1].lower()
    return ext if ext in [".wav", ".mp3", ".aac", ".ogg", ".flac", ".m4a"] else ".mp3"

def download_renan_file(url, save_path, dp_id):
    if os.path.exists(save_path):
        return {
            "dp_id": dp_id, "status": "already_exists",
            "file_size_kb": round(os.path.getsize(save_path) / 1024, 1),
            "md5": None, "error": None
        }
    try:
        r = requests.get(url, timeout=TIMEOUT, headers=HEADERS)
        r.raise_for_status()

        ctype = r.headers.get("Content-Type", "")
        if "text/html" in ctype:
            return {
                "dp_id": dp_id, "status": "bad_content_type",
                "file_size_kb": 0, "md5": None,
                "error": f"Got HTML page — URL may be dead"
            }

        # ── Write to temp file so ffmpeg can decode any format ──
        ext      = get_ext_from_url(url)
        tmp_path = None
        wav_path = None
        try:
            with tempfile.NamedTemporaryFile(suffix=ext, delete=False) as tmp:
                tmp.write(r.content)
                tmp_path = tmp.name

            # Try direct write first (already valid audio bytes)
            # We just save the raw bytes — Stage 3 preprocessing
            # will resample/normalize. No need to decode here.
            with open(save_path, "wb") as f:
                f.write(r.content)

            return {
                "dp_id": dp_id, "status": "downloaded",
                "file_size_kb": round(len(r.content) / 1024, 1),
                "md5": hashlib.md5(r.content).hexdigest()[:8], "error": None
            }
        finally:
            if tmp_path and os.path.exists(tmp_path):
                os.unlink(tmp_path)

    except requests.exceptions.Timeout:
        return {"dp_id": dp_id, "status": "error", "file_size_kb": 0,
                "md5": None, "error": "TIMEOUT"}
    except requests.exceptions.HTTPError as e:
        return {"dp_id": dp_id, "status": "error", "file_size_kb": 0,
                "md5": None, "error": f"HTTP_{e.response.status_code}"}
    except Exception as e:
        return {"dp_id": dp_id, "status": "error", "file_size_kb": 0,
                "md5": None, "error": str(e)[:80]}

for _, row in tqdm(df_renan.iterrows(), total=len(df_renan), desc="Renan files"):
    result = download_renan_file(row["audio_url"], row["save_path"], row["dp_id"])
    result.update({
        "source":    "renan",
        "label":     row["label"],
        "nativity":  row["nativity_status"],
        "dialect":   row["language"],
        "save_path": row["save_path"],
    })
    manifest_records.append(result)
    time.sleep(SLEEP)

a_df = pd.DataFrame(manifest_records)
print(f"\nRenan download complete:")
print(f"   Downloaded     : {(a_df['status']=='downloaded').sum()}")
print(f"   Already existed: {(a_df['status']=='already_exists').sum()}")
print(f"   Failed         : {(a_df['status'].isin(['error','bad_content_type'])).sum()}")


# ════════════════════════════════════════════════════════════
# PART B — MOZILLA COMMON VOICE 24.0 via API (streaming)
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 58)
print("  PART B — Common Voice 24.0 Arabic via API (streaming)")
print("=" * 58)

cv_records = []

if CV_API_KEY == "YOUR_API_KEY_HERE":
    print("⚠️  CV_API_KEY not set — skipping Part B.")
else:
    print("Requesting presigned download URL from Mozilla API...")
    try:
        resp = requests.post(
            CV_API_URL,
            headers={
                "Authorization": f"Bearer {CV_API_KEY}",
                "Content-Type":  "application/json",
            },
            timeout=30,
        )
        resp.raise_for_status()
        download_url = resp.json()["downloadUrl"]
        print(f"✅ Got presigned URL")
    except requests.exceptions.HTTPError as e:
        print(f"❌ API error: HTTP {e.response.status_code} — {e.response.text[:200]}")
        download_url = None
    except KeyError:
        print(f"❌ 'downloadUrl' not in response: {resp.json()}")
        download_url = None
    except Exception as e:
        print(f"❌ Request failed: {e}")
        download_url = None

    if download_url:
        class StreamingTarWrapper(io.RawIOBase):
            def __init__(self, response):
                self._iter = response.iter_content(chunk_size=1024 * 256)
                self._buf  = b""
                self.bytes_read = 0
            def readable(self): return True
            def readinto(self, b):
                while len(self._buf) < len(b):
                    try:
                        chunk = next(self._iter)
                        self._buf += chunk
                        self.bytes_read += len(chunk)
                    except StopIteration:
                        break
                n = min(len(b), len(self._buf))
                b[:n] = self._buf[:n]
                self._buf = self._buf[n:]
                return n

        # Pass 1: extract train.tsv
        print("\nPass 1: Streaming archive to extract train.tsv...")
        df_filtered  = None
        clips_prefix = None

        stream_resp1 = requests.get(download_url, stream=True, timeout=120)
        stream_resp1.raise_for_status()
        wrapper1 = StreamingTarWrapper(stream_resp1)

        with tarfile.open(fileobj=wrapper1, mode="r|gz") as tar:
            for member in tar:
                if (clips_prefix is None and
                        member.name.endswith(".mp3") and
                        "/ar/clips/" in member.name):
                    clips_prefix = member.name.rsplit("/ar/clips/", 1)[0] + "/ar/clips/"
                    print(f"  Detected clips prefix: {clips_prefix}")

                if member.name.endswith("ar/train.tsv"):
                    print(f"  Found TSV: {member.name}")
                    f_obj     = tar.extractfile(member)
                    tsv_bytes = io.BytesIO(f_obj.read())
                    df_cv     = pd.read_csv(tsv_bytes, sep="\t")
                    print(f"  Train clips: {len(df_cv):,}")

                    df_cv["up_votes"]   = pd.to_numeric(df_cv["up_votes"],   errors="coerce").fillna(0)
                    df_cv["down_votes"] = pd.to_numeric(df_cv["down_votes"], errors="coerce").fillna(0)
                    df_filtered = df_cv[
                        (df_cv["up_votes"]   >= CV_MIN_UPVOTES) &
                        (df_cv["down_votes"] == 0)
                    ].copy()
                    print(f"  After quality filter: {len(df_filtered):,}")

                    accent_col = "accents" if "accents" in df_filtered.columns else \
                                 "accent"  if "accent"  in df_filtered.columns else None
                    if accent_col is None:
                        df_filtered["_accent"] = "unspecified"
                        accent_col = "_accent"

                    df_filtered["accent"] = (
                        df_filtered[accent_col].fillna("unspecified").replace("", "unspecified")
                    )
                    accent_groups = df_filtered["accent"].unique()
                    n_groups      = len(accent_groups)
                    per_group     = max(1, CV_NATIVE_N // n_groups)
                    remainder     = CV_NATIVE_N - (per_group * n_groups)

                    sampled_parts = []
                    for i, accent in enumerate(accent_groups):
                        grp    = df_filtered[df_filtered["accent"] == accent]
                        n_take = min(per_group + (1 if i < remainder else 0), len(grp))
                        sampled_parts.append(grp.sample(n=n_take, random_state=42))

                    df_sample = pd.concat(sampled_parts, ignore_index=True)
                    print(f"  Sampled {len(df_sample)} clips")
                    break

        stream_resp1.close()

        if df_filtered is None:
            print("❌ Could not find ar/train.tsv in archive.")
        else:
            accent_to_dialect = {
                "saudi":       "Arabic_SA",
                "gulf":        "Arabic_QA",
                "egyptian":    "Arabic_EG",
                "levantine":   "Arabic_LEV",
                "moroccan":    "Arabic_MA",
                "unspecified": "Arabic_MSA",
            }

            clips_needed = {}
            for idx, row in df_sample.iterrows():
                accent_slug      = str(row["accent"]).replace(" ", "_").lower()
                out_fname        = f"cv_{accent_slug}_{row['path']}"
                out_path         = os.path.join(TRAIN_NATIVE, out_fname)
                clip_path_in_tar = (clips_prefix or "cv-corpus-24.0-2025-12-05/ar/clips/") + row["path"]
                dialect_label    = accent_to_dialect.get(str(row["accent"]).lower(), "Arabic_MSA")
                clips_needed[clip_path_in_tar] = {
                    "dp_id":     f"cv_{idx:06d}",
                    "out_path":  out_path,
                    "dialect":   dialect_label,
                    "cv_accent": row["accent"],
                    "up_votes":  int(row["up_votes"]),
                }

            already_on_disk = {k: v for k, v in clips_needed.items() if os.path.exists(v["out_path"])}
            to_fetch        = {k: v for k, v in clips_needed.items() if not os.path.exists(v["out_path"])}

            print(f"\nAlready on Drive : {len(already_on_disk)}")
            print(f"To fetch         : {len(to_fetch)}")

            for tar_path, info in already_on_disk.items():
                cv_records.append({
                    "dp_id": info["dp_id"], "source": "mozilla_common_voice_24",
                    "label": 1, "nativity": "Native", "dialect": info["dialect"],
                    "save_path": info["out_path"], "status": "already_exists",
                    "error": None, "cv_accent": info["cv_accent"], "up_votes": info["up_votes"],
                })

            if to_fetch:
                print(f"\nPass 2: Extracting {len(to_fetch)} clips...")
                try:
                    resp2 = requests.post(
                        CV_API_URL,
                        headers={"Authorization": f"Bearer {CV_API_KEY}",
                                 "Content-Type": "application/json"},
                        timeout=30,
                    )
                    resp2.raise_for_status()
                    download_url2 = resp2.json()["downloadUrl"]
                    print("✅ Fresh presigned URL obtained")
                except Exception as e:
                    print(f"⚠️  Could not refresh URL: {e}")
                    download_url2 = download_url

                stream_resp2 = requests.get(download_url2, stream=True, timeout=120)
                stream_resp2.raise_for_status()
                wrapper2  = StreamingTarWrapper(stream_resp2)
                extracted = 0
                errors    = 0
                pbar      = tqdm(total=len(to_fetch), desc="Extracting CV clips")

                with tarfile.open(fileobj=wrapper2, mode="r|gz") as tar:
                    for member in tar:
                        if member.name not in to_fetch:
                            continue
                        info  = to_fetch[member.name]
                        f_obj = tar.extractfile(member)
                        if f_obj is None:
                            cv_records.append({
                                "dp_id": info["dp_id"], "source": "mozilla_common_voice_24",
                                "label": 1, "nativity": "Native", "dialect": info["dialect"],
                                "save_path": None, "status": "error",
                                "error": "Empty tar member", "cv_accent": info["cv_accent"],
                                "up_votes": info["up_votes"],
                            })
                            errors += 1
                        else:
                            try:
                                with open(info["out_path"], "wb") as out_f:
                                    out_f.write(f_obj.read())
                                cv_records.append({
                                    "dp_id": info["dp_id"], "source": "mozilla_common_voice_24",
                                    "label": 1, "nativity": "Native", "dialect": info["dialect"],
                                    "save_path": info["out_path"], "status": "extracted",
                                    "error": None, "cv_accent": info["cv_accent"],
                                    "up_votes": info["up_votes"],
                                })
                                extracted += 1
                            except Exception as e:
                                cv_records.append({
                                    "dp_id": info["dp_id"], "source": "mozilla_common_voice_24",
                                    "label": 1, "nativity": "Native", "dialect": info["dialect"],
                                    "save_path": None, "status": "error",
                                    "error": str(e)[:80], "cv_accent": info["cv_accent"],
                                    "up_votes": info["up_votes"],
                                })
                                errors += 1
                        pbar.update(1)
                        if extracted + errors + len(already_on_disk) >= len(clips_needed):
                            break

                pbar.close()
                stream_resp2.close()
                print(f"\nExtracted: {extracted} | Errors: {errors}")

    manifest_records.extend(cv_records)


# ════════════════════════════════════════════════════════════
# PART C — MASTER MANIFEST
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 58)
print("  PART C — Master Manifest")
print("=" * 58)

manifest_df = pd.DataFrame(manifest_records)

usable = manifest_df[
    manifest_df["status"].isin(["downloaded", "already_exists", "extracted"]) &
    manifest_df["save_path"].notna()
].copy()

usable["file_on_disk"] = usable["save_path"].apply(os.path.exists)
ghost = (~usable["file_on_disk"]).sum()
if ghost:
    print(f"Removing {ghost} manifest entries where file missing on disk")
usable = usable[usable["file_on_disk"]].drop(columns=["file_on_disk"])

n_native    = (usable["label"] == 1).sum()
n_nonnative = (usable["label"] == 0).sum()
ratio       = n_native / n_nonnative if n_nonnative > 0 else float("inf")

print(f"\nFinal manifest:")
print(f"   Total usable files : {len(usable)}")
print(f"\n   By source:")
print(usable["source"].value_counts().to_string())
print(f"\n   By class:")
print(usable["nativity"].value_counts().to_string())
print(f"\n   By dialect:")
print(usable["dialect"].value_counts().to_string())
print(f"\n   Class balance — Native: {n_native} | Non-Native: {n_nonnative} | Ratio: {ratio:.1f}x")

manifest_df.to_csv(f"{OUTPUT_DIR}/stage2_manifest.csv", index=False)
print(f"\nSaved: {OUTPUT_DIR}/stage2_manifest.csv")
print("\n✅ Stage 2 complete.")
print("⏭️  Next: Run Cell 4 (Stage 3 — Preprocessing)")

  PART A — Downloading Renan Training Audio (178 files)


Renan files:   0%|          | 0/178 [00:00<?, ?it/s]


Renan download complete:
   Downloaded     : 178
   Already existed: 0
   Failed         : 0

  PART B — Common Voice 24.0 Arabic via API (streaming)
Requesting presigned download URL from Mozilla API...
✅ Got presigned URL

Pass 1: Streaming archive to extract train.tsv...
  Found TSV: cv-corpus-24.0-2025-12-05/ar/train.tsv
  Train clips: 28,870
  After quality filter: 20,100
  Sampled 200 clips

Already on Drive : 0
To fetch         : 200

Pass 2: Extracting 200 clips...
✅ Fresh presigned URL obtained


Extracting CV clips:   0%|          | 0/200 [00:00<?, ?it/s]


Extracted: 200 | Errors: 0

  PART C — Master Manifest

Final manifest:
   Total usable files : 378

   By source:
source
mozilla_common_voice_24    200
renan                      178

   By class:
nativity
Native        314
Non-Native     64

   By dialect:
dialect
Arabic_MSA    200
Arabic_SA      63
Arabic_QA      56
Arabic_AE      38
Unknown_NN     18
Arabic_KW       3

   Class balance — Native: 314 | Non-Native: 64 | Ratio: 4.9x

Saved: /content/drive/MyDrive/team_databaes/outputs/stage2_manifest.csv

✅ Stage 2 complete.
⏭️  Next: Run Cell 4 (Stage 3 — Preprocessing)


In [6]:
# ============================================================
# CELL 4 — STAGE 3: PREPROCESSING (UPDATED)
# Changes from original:
#   - load_audio() uses ffmpeg via temp file on disk
#     instead of librosa BytesIO — handles AAC, WAV, MP3
#     and any other format correctly
# Everything else is identical to the original Stage 3.
# ============================================================

import os, warnings, subprocess, tempfile
import numpy as np, pandas as pd
import librosa, soundfile as sf, noisereduce as nr
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore")

PROJECT_ROOT  = "/content/drive/MyDrive/team_databaes"
MANIFEST_IN   = f"{PROJECT_ROOT}/outputs/stage2_manifest.csv"
MANIFEST_OUT  = f"{PROJECT_ROOT}/outputs/stage3_manifest.csv"
PREPROC_DIR   = f"{PROJECT_ROOT}/data/preprocessed"

TARGET_SR    = 16000
TARGET_RMS   = 0.05
MIN_DURATION = 3.0
VAD_TOP_DB   = 20
NR_PROP      = 0.75

os.makedirs(f"{PREPROC_DIR}/native",     exist_ok=True)
os.makedirs(f"{PREPROC_DIR}/non_native", exist_ok=True)

# ── Preprocessing functions ───────────────────────────────────

def load_audio(filepath):
    """
    Load audio via ffmpeg temp-file conversion.
    Handles MP3, WAV, AAC, and any other format ffmpeg supports.
    """
    wav_tmp = tempfile.mktemp(suffix=".wav")
    try:
        subprocess.run(
            ["ffmpeg", "-y", "-i", filepath, "-ar", "16000", "-ac", "1", wav_tmp],
            capture_output=True, check=True
        )
        y, sr = librosa.load(wav_tmp, sr=TARGET_SR, mono=True)
    finally:
        if os.path.exists(wav_tmp):
            os.unlink(wav_tmp)
    return y, sr

def rms_normalize(y, target_rms=TARGET_RMS):
    current_rms = np.sqrt(np.mean(y ** 2))
    if current_rms < 1e-9:
        return y
    return y * (target_rms / current_rms)

def vad_trim_edges(y, top_db=VAD_TOP_DB):
    y_trimmed, _ = librosa.effects.trim(y, top_db=top_db)
    return y_trimmed

def reduce_noise(y, sr, prop_decrease=NR_PROP):
    return nr.reduce_noise(y=y, sr=sr, prop_decrease=prop_decrease, stationary=True)

def preprocess_file(filepath, label, dp_id):
    result = {
        "dp_id":           dp_id,
        "original_path":   filepath,
        "preproc_status":  None,
        "preproc_path":    None,
        "duration_before": None,
        "duration_after":  None,
        "dropped_reason":  None,
    }
    try:
        y, sr = load_audio(filepath)
        result["duration_before"] = round(len(y) / sr, 2)

        y = rms_normalize(y)
        y = vad_trim_edges(y)
        y = reduce_noise(y, sr)

        duration_after = len(y) / sr
        result["duration_after"] = round(duration_after, 2)

        if duration_after < MIN_DURATION:
            result["preproc_status"] = "dropped_too_short"
            result["dropped_reason"] = f"Only {duration_after:.1f}s after VAD trim"
            return result

        subdir   = "native" if label == 1 else "non_native"
        out_path = os.path.join(PREPROC_DIR, subdir, f"{dp_id}.wav")
        sf.write(out_path, y, TARGET_SR, subtype="PCM_16")

        result["preproc_status"] = "ok"
        result["preproc_path"]   = out_path

    except Exception as e:
        result["preproc_status"] = "error"
        result["dropped_reason"] = str(e)[:120]

    return result

# ── Load manifest ─────────────────────────────────────────────
print("=" * 58)
print("  STAGE 3 — PREPROCESSING")
print("=" * 58)

df = pd.read_csv(MANIFEST_IN)

processable = df[
    df["status"].isin(["downloaded", "already_exists", "extracted"]) &
    df["save_path"].notna()
].copy()

print(f"\nTotal files in manifest : {len(df)}")
print(f"Files to process        : {len(processable)}")
print(f"Target SR               : {TARGET_SR} Hz")
print(f"Target RMS              : {TARGET_RMS}")
print(f"VAD top_db              : {VAD_TOP_DB}")
print(f"Noise reduction         : {int(NR_PROP*100)}%")
print(f"Min duration gate       : {MIN_DURATION}s\n")

# ── RESUME: skip files already preprocessed on Drive ─────────
def get_expected_out_path(row):
    subdir = "native" if row["label"] == 1 else "non_native"
    return os.path.join(PREPROC_DIR, subdir, f"{row['dp_id']}.wav")

processable["expected_out"] = processable.apply(get_expected_out_path, axis=1)
already_done = processable[processable["expected_out"].apply(os.path.exists)]
todo         = processable[~processable["expected_out"].apply(os.path.exists)]

if len(already_done) > 0:
    print(f"⏭️  Skipping {len(already_done)} files already preprocessed (resume)")
print(f"Processing {len(todo)} files...\n")

# ── Run preprocessing ─────────────────────────────────────────
preproc_results = []

for _, row in already_done.iterrows():
    preproc_results.append({
        "dp_id":           row["dp_id"],
        "original_path":   row["save_path"],
        "preproc_status":  "ok",
        "preproc_path":    row["expected_out"],
        "duration_before": None,
        "duration_after":  None,
        "dropped_reason":  None,
    })

for _, row in tqdm(todo.iterrows(), total=len(todo), desc="Preprocessing"):
    res = preprocess_file(
        filepath = row["save_path"],
        label    = row["label"],
        dp_id    = row["dp_id"],
    )
    preproc_results.append(res)

# ── Merge into manifest ───────────────────────────────────────
preproc_df = pd.DataFrame(preproc_results)
df_out     = processable.merge(preproc_df, on="dp_id", how="left")

ok      = df_out[df_out["preproc_status"] == "ok"]
dropped = df_out[df_out["preproc_status"] == "dropped_too_short"]
errors  = df_out[df_out["preproc_status"] == "error"]

def get_duration(path):
    try:
        info = sf.info(path)
        return round(info.duration, 2)
    except Exception:
        return None

ok_with_dur  = ok.copy()
mask_no_dur  = ok_with_dur["duration_after"].isna()
if mask_no_dur.sum() > 0:
    ok_with_dur.loc[mask_no_dur, "duration_after"] = \
        ok_with_dur.loc[mask_no_dur, "preproc_path"].apply(get_duration)
    df_out.update(ok_with_dur[["dp_id","duration_after"]].set_index("dp_id"))

ok = df_out[df_out["preproc_status"] == "ok"]

# ── Report ────────────────────────────────────────────────────
print("\n" + "=" * 58)
print("  STAGE 3 — REPORT")
print("=" * 58)
print(f"\n   Successfully preprocessed : {len(ok)}")
print(f"   Dropped (too short)       : {len(dropped)}")
print(f"   Errors                    : {len(errors)}")

if len(dropped) > 0:
    cols = [c for c in ["dp_id","nativity","dialect","dropped_reason"] if c in dropped.columns]
    print(f"\n   Dropped files:")
    print(dropped[cols].to_string(index=False))

if len(errors) > 0:
    cols = [c for c in ["dp_id","nativity","dropped_reason"] if c in errors.columns]
    print(f"\n   Error files:")
    print(errors[cols].to_string(index=False))

ok_dur = ok["duration_after"].dropna()
if len(ok_dur) > 0:
    print(f"\n   Duration after VAD trim:")
    print(f"     mean={ok_dur.mean():.1f}s  min={ok_dur.min():.1f}s  max={ok_dur.max():.1f}s")

print(f"\n   Usable files by class:")
if "nativity" in ok.columns:
    print(ok["nativity"].value_counts().to_string())

est_segs = ok_dur.apply(lambda d: max(0, int((d - 3.0) / 1.0) + 1)).sum()
print(f"\n   Estimated 3s segments (Stage 4 preview): ~{int(est_segs):,}")
print(f"   (3s window, 1s hop, based on trimmed durations)")

# ── Save manifest ─────────────────────────────────────────────
df_out.to_csv(MANIFEST_OUT, index=False)
print(f"\nSaved: {MANIFEST_OUT}")
print("\n✅ Stage 3 complete.")
print("⏭️  Next: Run Cell 5 (Stage 4 — Splitting)")


  STAGE 3 — PREPROCESSING

Total files in manifest : 378
Files to process        : 378
Target SR               : 16000 Hz
Target RMS              : 0.05
VAD top_db              : 20
Noise reduction         : 75%
Min duration gate       : 3.0s

Processing 378 files...



Preprocessing:   0%|          | 0/378 [00:00<?, ?it/s]


  STAGE 3 — REPORT

   Successfully preprocessed : 234
   Dropped (too short)       : 144
   Errors                    : 0

   Dropped files:
    dp_id nativity    dialect           dropped_reason
cv_000199   Native Arabic_MSA Only 2.0s after VAD trim
cv_000112   Native Arabic_MSA Only 1.4s after VAD trim
cv_000156   Native Arabic_MSA Only 1.8s after VAD trim
cv_000014   Native Arabic_MSA Only 2.3s after VAD trim
cv_000138   Native Arabic_MSA Only 1.4s after VAD trim
cv_000056   Native Arabic_MSA Only 2.2s after VAD trim
cv_000189   Native Arabic_MSA Only 1.1s after VAD trim
cv_000064   Native Arabic_MSA Only 1.9s after VAD trim
cv_000186   Native Arabic_MSA Only 1.4s after VAD trim
cv_000088   Native Arabic_MSA Only 2.4s after VAD trim
cv_000027   Native Arabic_MSA Only 2.9s after VAD trim
cv_000178   Native Arabic_MSA Only 1.4s after VAD trim
cv_000080   Native Arabic_MSA Only 1.1s after VAD trim
cv_000122   Native Arabic_MSA Only 2.1s after VAD trim
cv_000023   Native Arabic_MSA On

In [7]:
# ============================================================
# STAGE 4: AUDIO SPLITTING
# Cuts every preprocessed WAV into overlapping 3s segments
# with a 1s hop. Each segment inherits parent file's label.
#
# COLAB SETUP (run this cell first in your notebook):
#   from google.colab import drive
#   drive.mount('/content/drive')
# ============================================================
import os
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from tqdm.notebook import tqdm   # notebook-friendly progress bars

# ── Google Drive path config ─────────────────────────────────────
# Change this to match your actual Drive folder name
PROJECT_ROOT = "/content/drive/MyDrive/team_databaes"

MANIFEST_IN  = f"{PROJECT_ROOT}/outputs/stage3_manifest.csv"
MANIFEST_OUT = f"{PROJECT_ROOT}/outputs/stage4_manifest.csv"
SEGMENTS_DIR = f"{PROJECT_ROOT}/data/segments"

WINDOW_SEC = 3.0
HOP_SEC    = 1.0
TARGET_SR  = 16000

os.makedirs(f"{SEGMENTS_DIR}/native",     exist_ok=True)
os.makedirs(f"{SEGMENTS_DIR}/non_native", exist_ok=True)

df     = pd.read_csv(MANIFEST_IN)
usable = df[df["preproc_status"] == "ok"].copy()
print(f"Usable files: {len(usable)}")

window_samples = int(WINDOW_SEC * TARGET_SR)   # 48000 samples
hop_samples    = int(HOP_SEC    * TARGET_SR)   # 16000 samples

segment_records = []

for _, row in tqdm(usable.iterrows(), total=len(usable), desc="Splitting"):
    try:
        y, sr = librosa.load(row["preproc_path"], sr=TARGET_SR, mono=True)
    except Exception as e:
        print(f"  Load error {row['dp_id']}: {e}")
        continue

    n_segments = max(0, (len(y) - window_samples) // hop_samples + 1)
    subdir     = "native" if row["label"] == 1 else "non_native"

    for i in range(n_segments):
        start   = i * hop_samples
        end     = start + window_samples
        segment = y[start:end]

        if len(segment) < window_samples:
            segment = np.pad(segment, (0, window_samples - len(segment)))

        seg_id   = f"{row['dp_id']}_seg{i:04d}"
        seg_path = os.path.join(SEGMENTS_DIR, subdir, f"{seg_id}.wav")

        sf.write(seg_path, segment, TARGET_SR, subtype="PCM_16")

        segment_records.append({
            "seg_id":    seg_id,
            "parent_id": row["dp_id"],
            "seg_index": i,
            "label":     row["label"],
            "nativity":  row["nativity"],
            "dialect":   row["dialect"],
            "source":    row["source"],
            "seg_path":  seg_path,
            "start_sec": round(i * HOP_SEC, 2),
            "end_sec":   round(i * HOP_SEC + WINDOW_SEC, 2),
        })

seg_df = pd.DataFrame(segment_records)
seg_df.to_csv(MANIFEST_OUT, index=False)

print(f"\nTotal segments created : {len(seg_df)}")
print(f"Native segments        : {(seg_df['label']==1).sum()}")
print(f"Non-native segments    : {(seg_df['label']==0).sum()}")
print(f"Saved: {MANIFEST_OUT}")

Usable files: 234


Splitting:   0%|          | 0/234 [00:00<?, ?it/s]


Total segments created : 8061
Native segments        : 4786
Non-native segments    : 3275
Saved: /content/drive/MyDrive/team_databaes/outputs/stage4_manifest.csv


In [8]:
# ============================================================
# STAGE 5: DATA AUGMENTATION (training segments only)
# Non-native (minority): 5 augmentations
# Native (majority):     1 augmentation (time stretch only)
# NEVER augment test data.
#
# COLAB NOTE: Augmentation is CPU-heavy. On Colab free tier
# (~17k segments × 5 augs) expect ~45–90 min. Colab Pro/T4
# does not speed this up — librosa runs on CPU only.
# Consider running overnight or in sections using RESUME below.
# ============================================================
import os
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from tqdm.notebook import tqdm

# ── Google Drive path config ─────────────────────────────────────
PROJECT_ROOT = "/content/drive/MyDrive/team_databaes"

MANIFEST_IN  = f"{PROJECT_ROOT}/outputs/stage4_manifest.csv"
MANIFEST_OUT = f"{PROJECT_ROOT}/outputs/stage5_manifest.csv"
AUG_DIR      = f"{PROJECT_ROOT}/data/augmented"
TARGET_SR    = 16000
RANDOM_SEED  = 42
np.random.seed(RANDOM_SEED)

os.makedirs(f"{AUG_DIR}/native",     exist_ok=True)
os.makedirs(f"{AUG_DIR}/non_native", exist_ok=True)

seg_df = pd.read_csv(MANIFEST_IN)
print(f"Segments to augment: {len(seg_df)}")

# ── RESUME SUPPORT ───────────────────────────────────────────────
# If the Colab session disconnects mid-run, re-running will skip
# already-completed segments instead of starting over.
already_done = set()
if os.path.exists(MANIFEST_OUT):
    done_df      = pd.read_csv(MANIFEST_OUT)
    already_done = set(done_df[done_df["aug_type"] != "original"]["orig_seg"].dropna())
    print(f"Resuming — {len(already_done)} segments already augmented, skipping.")

# ── Augmentation functions ───────────────────────────────────────
def time_stretch(y, rate):
    return librosa.effects.time_stretch(y, rate=rate)

def pitch_shift(y, sr, steps):
    return librosa.effects.pitch_shift(y, sr=sr, n_steps=steps)

def add_noise(y, sigma=0.005):
    noise = np.random.randn(len(y)) * sigma
    return np.clip(y + noise, -1.0, 1.0)

def save_augmented(y, sr, seg_id, aug_name, label):
    subdir   = "native" if label == 1 else "non_native"
    filename = f"{seg_id}_{aug_name}.wav"
    out_path = os.path.join(AUG_DIR, subdir, filename)
    sf.write(out_path, y, sr, subtype="PCM_16")
    return out_path

# ── Apply augmentations ──────────────────────────────────────────
aug_records = []

for _, row in tqdm(seg_df.iterrows(), total=len(seg_df), desc="Augmenting"):
    # Skip if already processed (resume support)
    if row["seg_id"] in already_done:
        continue

    try:
        y, sr = librosa.load(row["seg_path"], sr=TARGET_SR, mono=True)
    except Exception as e:
        print(f"  Load error {row['seg_id']}: {e}")
        continue

    augmentations = []
    if row["label"] == 0:
        # Non-native (minority class): all 3 augmentation types (5 variants)
        augmentations = [
            ("ts_slow",  time_stretch(y, 0.9)),
            ("ts_fast",  time_stretch(y, 1.1)),
            ("ps_up",    pitch_shift(y, sr, +2)),
            ("ps_down",  pitch_shift(y, sr, -2)),
            ("noise",    add_noise(y)),
        ]
    else:
        # Native (majority class): time stretch only
        augmentations = [
            ("ts_slow",  time_stretch(y, 0.9)),
        ]

    for aug_name, y_aug in augmentations:
        if len(y_aug) > 48000:
            y_aug = y_aug[:48000]
        elif len(y_aug) < 48000:
            y_aug = np.pad(y_aug, (0, 48000 - len(y_aug)))

        out_path = save_augmented(y_aug, sr, row["seg_id"], aug_name, row["label"])
        aug_records.append({
            "seg_id":    f"{row['seg_id']}_{aug_name}",
            "parent_id": row["parent_id"],
            "seg_index": row["seg_index"],
            "label":     row["label"],
            "nativity":  row["nativity"],
            "dialect":   row["dialect"],
            "source":    "augmented",
            "seg_path":  out_path,
            "aug_type":  aug_name,
            "orig_seg":  row["seg_id"],
        })

# ── Combine originals + new augmentations ────────────────────────
aug_df = pd.DataFrame(aug_records)

# If resuming, load existing aug records and append only new ones
if os.path.exists(MANIFEST_OUT) and len(already_done) > 0:
    existing = pd.read_csv(MANIFEST_OUT)
    aug_df   = pd.concat([existing, aug_df], ignore_index=True)
    # Rebuild: originals are from stage4_manifest, augs from aug_df
    originals = seg_df.assign(aug_type="original", orig_seg=None)
    all_segs  = pd.concat(
        [originals, aug_df[aug_df["aug_type"] != "original"]],
        ignore_index=True
    )
else:
    originals = seg_df.assign(aug_type="original", orig_seg=None)
    all_segs  = pd.concat([originals, aug_df], ignore_index=True)

all_segs.to_csv(MANIFEST_OUT, index=False)

print(f"\nOriginal segments  : {len(seg_df)}")
print(f"Augmented segments : {(all_segs['aug_type'] != 'original').sum()}")
print(f"Total segments     : {len(all_segs)}")
print(f"Native total       : {(all_segs['label']==1).sum()}")
print(f"Non-native total   : {(all_segs['label']==0).sum()}")
print(f"Saved: {MANIFEST_OUT}")

Segments to augment: 8061


Augmenting:   0%|          | 0/8061 [00:00<?, ?it/s]


Original segments  : 8061
Augmented segments : 21161
Total segments     : 29222
Native total       : 9572
Non-native total   : 19650
Saved: /content/drive/MyDrive/team_databaes/outputs/stage5_manifest.csv


In [9]:
# ============================================================
# STAGE 6A: wav2vec2 EMBEDDING EXTRACTION
# Model: jonatasgrosman/wav2vec2-large-xlsr-53-arabic (frozen)
# Output: (N_segments, 768) numpy array via mean pooling
#
# COLAB NOTES:
#   - Runtime > GPU is STRONGLY recommended (Runtime → Change runtime type)
#   - Model downloads ~1.2GB on first run, cached in /root/.cache/
#     BUT Colab cache clears on session reset — model re-downloads each new session
#     To persist the cache: set HF_HOME to a Drive path (see config below)
#   - BATCH_SIZE=16 works on T4 (16GB VRAM). Use 8 on free tier (12GB)
#   - Checkpoint saves every CHECKPOINT_EVERY batches so a session
#     disconnect doesn't lose all progress
# ============================================================
import os
import numpy as np
import pandas as pd
import torch
import librosa
from transformers import Wav2Vec2Model, Wav2Vec2Processor
from tqdm.notebook import tqdm

# ── Google Drive path config ─────────────────────────────────────
PROJECT_ROOT = "/content/drive/MyDrive/team_databaes"

MANIFEST_IN       = f"{PROJECT_ROOT}/outputs/stage5_manifest.csv"
EMBED_DIR         = f"{PROJECT_ROOT}/data/embeddings"
TARGET_SR         = 16000
BATCH_SIZE        = 16        # reduce to 8 if CUDA OOM (free tier Colab)
CHECKPOINT_EVERY  = 50        # save partial results every N batches
DEVICE            = "cuda" if torch.cuda.is_available() else "cpu"

# ── Persist HuggingFace model cache to Drive (avoids re-downloading each session)
HF_CACHE = f"{PROJECT_ROOT}/hf_cache"
os.environ["HF_HOME"]            = HF_CACHE
os.environ["TRANSFORMERS_CACHE"] = HF_CACHE
os.makedirs(HF_CACHE,  exist_ok=True)
os.makedirs(EMBED_DIR, exist_ok=True)

print(f"Using device: {DEVICE}")
if DEVICE == "cpu":
    print("WARNING: No GPU detected. Embedding extraction will be very slow on CPU.")
    print("Go to Runtime → Change runtime type → GPU (T4)")

# ── Load model ───────────────────────────────────────────────────
print("Loading wav2vec2 model (downloads ~1.2GB if not cached)...")
processor = Wav2Vec2Processor.from_pretrained("jonatasgrosman/wav2vec2-large-xlsr-53-arabic")
model     = Wav2Vec2Model.from_pretrained("jonatasgrosman/wav2vec2-large-xlsr-53-arabic")
model     = model.to(DEVICE)
model.eval()
print("Model loaded.")

seg_df = pd.read_csv(MANIFEST_IN)
print(f"Total segments to embed: {len(seg_df)}")

# ── CHECKPOINT / RESUME ──────────────────────────────────────────
checkpoint_embeds_path  = f"{EMBED_DIR}/wav2vec2_embeddings_partial.npy"
checkpoint_segids_path  = f"{EMBED_DIR}/wav2vec2_seg_ids_partial.npy"
start_batch = 0

all_embeddings = []
all_seg_ids    = []

if os.path.exists(checkpoint_embeds_path) and os.path.exists(checkpoint_segids_path):
    prev_embeds  = np.load(checkpoint_embeds_path)
    prev_seg_ids = np.load(checkpoint_segids_path, allow_pickle=True)
    all_embeddings.append(prev_embeds)
    all_seg_ids.extend(prev_seg_ids.tolist())
    start_batch  = (len(prev_seg_ids) // BATCH_SIZE)
    print(f"Resuming from batch {start_batch} ({len(prev_seg_ids)} segments already done)")

# ── Embedding extraction ─────────────────────────────────────────
def extract_embedding_batch(paths):
    waveforms = []
    for path in paths:
        y, _ = librosa.load(path, sr=TARGET_SR, mono=True)
        if len(y) > 48000:
            y = y[:48000]
        elif len(y) < 48000:
            y = np.pad(y, (0, 48000 - len(y)))
        waveforms.append(y)

    inputs = processor(
        waveforms,
        sampling_rate=TARGET_SR,
        return_tensors="pt",
        padding=True
    )
    input_values = inputs.input_values.to(DEVICE)

    with torch.no_grad():
        outputs    = model(input_values)
        embeddings = outputs.last_hidden_state.mean(dim=1)   # (batch, 768)

    return embeddings.cpu().numpy()

paths   = seg_df["seg_path"].tolist()
seg_ids = seg_df["seg_id"].tolist()
errors  = []

# Skip already-processed batches
paths_todo   = paths[start_batch * BATCH_SIZE:]
seg_ids_todo = seg_ids[start_batch * BATCH_SIZE:]

for i in tqdm(range(0, len(paths_todo), BATCH_SIZE), desc="Extracting embeddings"):
    batch_paths   = paths_todo[i : i + BATCH_SIZE]
    batch_seg_ids = seg_ids_todo[i : i + BATCH_SIZE]

    try:
        embeds = extract_embedding_batch(batch_paths)
        all_embeddings.append(embeds)
        all_seg_ids.extend(batch_seg_ids)
    except RuntimeError as e:
        if "CUDA out of memory" in str(e):
            print(f"\nOOM at batch {i} — try reducing BATCH_SIZE to {BATCH_SIZE // 2}")
            torch.cuda.empty_cache()
            raise
        print(f"  Batch {i} error: {e}")
        errors.append({"batch_start": i, "error": str(e)})
        all_embeddings.append(np.zeros((len(batch_paths), 768)))
        all_seg_ids.extend(batch_seg_ids)

    # Save checkpoint periodically (protects against Colab session drops)
    if (i // BATCH_SIZE + 1) % CHECKPOINT_EVERY == 0:
        partial_embeds = np.vstack(all_embeddings)
        np.save(checkpoint_embeds_path, partial_embeds)
        np.save(checkpoint_segids_path, np.array(all_seg_ids))
        print(f"  Checkpoint saved at batch {i // BATCH_SIZE + 1} ({len(all_seg_ids)} segments)")

# ── Final save ───────────────────────────────────────────────────
X_wav2vec = np.vstack(all_embeddings)   # (N_segments, 768)
np.save(f"{EMBED_DIR}/wav2vec2_embeddings.npy", X_wav2vec)
np.save(f"{EMBED_DIR}/wav2vec2_seg_ids.npy",    np.array(all_seg_ids))

# Clean up checkpoint files
for f in [checkpoint_embeds_path, checkpoint_segids_path]:
    if os.path.exists(f):
        os.remove(f)

print(f"\nEmbeddings shape : {X_wav2vec.shape}")
print(f"Saved: {EMBED_DIR}/wav2vec2_embeddings.npy")
if errors:
    print(f"WARNING: Errors in {len(errors)} batches")

Using device: cuda
Loading wav2vec2 model (downloads ~1.2GB if not cached)...


preprocessor_config.json:   0%|          | 0.00/158 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json:   0%|          | 0.00/507 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Wav2Vec2Model LOAD REPORT from: jonatasgrosman/wav2vec2-large-xlsr-53-arabic
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 
lm_head.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded.
Total segments to embed: 29222


Extracting embeddings:   0%|          | 0/1827 [00:00<?, ?it/s]

  Checkpoint saved at batch 50 (800 segments)
  Checkpoint saved at batch 100 (1600 segments)
  Checkpoint saved at batch 150 (2400 segments)
  Checkpoint saved at batch 200 (3200 segments)
  Checkpoint saved at batch 250 (4000 segments)
  Checkpoint saved at batch 300 (4800 segments)
  Checkpoint saved at batch 350 (5600 segments)
  Checkpoint saved at batch 400 (6400 segments)
  Checkpoint saved at batch 450 (7200 segments)
  Checkpoint saved at batch 500 (8000 segments)
  Checkpoint saved at batch 550 (8800 segments)
  Checkpoint saved at batch 600 (9600 segments)
  Checkpoint saved at batch 650 (10400 segments)
  Checkpoint saved at batch 700 (11200 segments)
  Checkpoint saved at batch 750 (12000 segments)
  Checkpoint saved at batch 800 (12800 segments)
  Checkpoint saved at batch 850 (13600 segments)
  Checkpoint saved at batch 900 (14400 segments)
  Checkpoint saved at batch 950 (15200 segments)
  Checkpoint saved at batch 1000 (16000 segments)
  Checkpoint saved at batch 1050 

In [10]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Loaded parameters: {total_params:,}")
# Expect ~300M for wav2vec2-large — confirms full encoder loaded correctly

Loaded parameters: 315,438,720


In [11]:
# ============================================================
# STAGE 6B: PROSODIC FEATURE EXTRACTION
# Runs on UTTERANCE level (stage3_manifest).
# All segments from the same recording share the same 6-dim
# prosodic vector (looked up by parent_id in Stage 7).
#
# COLAB NOTES:
#   - parselmouth must be installed: pip install praat-parselmouth
#   - This runs on CPU — no GPU benefit here
#   - Can be run in parallel with Stage 6A (different inputs/outputs)
# ============================================================
import os
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

# Install parselmouth if not already installed (safe to re-run)
try:
    import parselmouth
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "praat-parselmouth", "-q"], check=True)
    import parselmouth

# ── Google Drive path config ─────────────────────────────────────
PROJECT_ROOT = "/content/drive/MyDrive/team_databaes"

MANIFEST_IN  = f"{PROJECT_ROOT}/outputs/stage3_manifest.csv"
PROSODIC_OUT = f"{PROJECT_ROOT}/data/embeddings/prosodic_features.csv"

os.makedirs(f"{PROJECT_ROOT}/data/embeddings", exist_ok=True)


def extract_prosodic_features(audio_path):
    """
    Extract 6 prosodic features from a full utterance WAV file.
    Returns np.array of shape (6,) or zeros on failure.
    Features: [f0_mean, f0_var, speaking_rate, pause_freq,
               pause_dur_mean, npvi]
    """
    try:
        sound    = parselmouth.Sound(audio_path)
        duration = sound.end_time - sound.start_time

        # ── F0 mean and variance ──────────────────────────────
        pitch     = sound.to_pitch(time_step=0.01, pitch_floor=75, pitch_ceiling=600)
        f0_vals   = pitch.selected_array['frequency']
        f0_voiced = f0_vals[f0_vals > 0]

        f0_mean = float(np.mean(f0_voiced)) if len(f0_voiced) > 0 else 0.0
        f0_var  = float(np.var(f0_voiced))  if len(f0_voiced) > 0 else 0.0

        # ── Speaking rate (syllable-proxy peaks/sec) ──────────
        intensity  = sound.to_intensity(minimum_pitch=75)
        int_values = intensity.values.flatten()
        int_mean   = float(np.mean(int_values))

        peaks = 0
        for j in range(1, len(int_values) - 1):
            if (int_values[j] > int_mean and
                    int_values[j] > int_values[j-1] and
                    int_values[j] > int_values[j+1]):
                peaks += 1

        speaking_rate = peaks / duration if duration > 0 else 0.0

        # ── Pause frequency and mean pause duration ───────────
        pause_threshold_sec = 0.2

        f0_times  = [pitch.get_time_from_frame_number(i+1)
                     for i in range(pitch.get_number_of_frames())]
        is_voiced = [pitch.get_value_at_time(t) is not None and
                     pitch.get_value_at_time(t) > 0
                     for t in f0_times]

        pauses      = []
        in_pause    = False
        pause_start = 0.0

        for t, voiced in zip(f0_times, is_voiced):
            if not voiced and not in_pause:
                in_pause    = True
                pause_start = t
            elif voiced and in_pause:
                pause_dur = t - pause_start
                if pause_dur >= pause_threshold_sec:
                    pauses.append(pause_dur)
                in_pause = False

        pause_freq     = len(pauses) / duration if duration > 0 else 0.0
        pause_dur_mean = float(np.mean(pauses)) if pauses else 0.0

        # ── nPVI ──────────────────────────────────────────────
        vowel_durations = []
        in_vowel        = False
        vowel_start     = 0.0

        for t, voiced in zip(f0_times, is_voiced):
            if voiced and not in_vowel:
                in_vowel    = True
                vowel_start = t
            elif not voiced and in_vowel:
                vowel_dur = t - vowel_start
                if vowel_dur >= 0.03:
                    vowel_durations.append(vowel_dur)
                in_vowel = False

        if len(vowel_durations) >= 2:
            durs  = np.array(vowel_durations)
            dk    = durs[:-1]
            dk1   = durs[1:]
            denom = (dk + dk1) / 2
            valid = denom > 0
            npvi  = float(np.mean(np.abs(dk[valid] - dk1[valid]) / denom[valid]) * 100)
        else:
            npvi = 0.0

        return np.array([f0_mean, f0_var, speaking_rate,
                         pause_freq, pause_dur_mean, npvi], dtype=np.float32)

    except Exception as e:
        print(f"  Prosodic extraction failed for {audio_path}: {e}")
        return np.zeros(6, dtype=np.float32)


# ── Run extraction ───────────────────────────────────────────────
df     = pd.read_csv(MANIFEST_IN)
usable = df[df["preproc_status"] == "ok"].copy()
print(f"Extracting prosodic features from {len(usable)} utterances...")

records = []
for _, row in tqdm(usable.iterrows(), total=len(usable), desc="Prosodic features"):
    features = extract_prosodic_features(row["preproc_path"])
    records.append({
        "dp_id":          row["dp_id"],
        "label":          row["label"],
        "f0_mean":        features[0],
        "f0_var":         features[1],
        "speaking_rate":  features[2],
        "pause_freq":     features[3],
        "pause_dur_mean": features[4],
        "npvi":           features[5],
    })

prosodic_df = pd.DataFrame(records)
prosodic_df.to_csv(PROSODIC_OUT, index=False)

print(f"\nProsodic features shape : {prosodic_df.shape}")
print(f"Saved: {PROSODIC_OUT}")

print("\nFeature means by class:")
print(prosodic_df.groupby("label")[
    ["f0_mean","f0_var","speaking_rate","pause_freq","pause_dur_mean","npvi"]
].mean().round(3).to_string())

Extracting prosodic features from 234 utterances...


Prosodic features:   0%|          | 0/234 [00:00<?, ?it/s]


Prosodic features shape : (234, 8)
Saved: /content/drive/MyDrive/team_databaes/data/embeddings/prosodic_features.csv

Feature means by class:
          f0_mean       f0_var  speaking_rate  pause_freq  pause_dur_mean       npvi
label                                                                               
0      187.987000  3344.006104          6.552       0.515           0.550  77.996002
1      203.033997  2805.806885          6.767       0.405           0.417  82.788002


In [12]:
# ============================================================
# STAGE 7: FEATURE FUSION
# Concatenates wav2vec2 (768-dim) + prosodic (6-dim) per segment
# → 774-dim feature vector.
# StandardScaler fitted on TRAIN only, applied to test.
# Group-level split by parent_id — no data leakage.
#
# COLAB NOTES:
#   - Pure numpy/sklearn — no GPU needed, runs fast on CPU
#   - Loads .npy files from Drive (may be slow if files are large;
#     consider copying to /content/ first for faster I/O)
# ============================================================
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
import joblib

# ── Google Drive path config ─────────────────────────────────────
PROJECT_ROOT = "/content/drive/MyDrive/team_databaes"

EMBED_DIR    = f"{PROJECT_ROOT}/data/embeddings"
MANIFEST_IN  = f"{PROJECT_ROOT}/outputs/stage5_manifest.csv"
FEATURES_OUT = f"{PROJECT_ROOT}/outputs/stage7_features.npz"

os.makedirs(f"{PROJECT_ROOT}/outputs", exist_ok=True)

# ── Optional: copy .npy files to local /content/ for faster I/O ──
# Uncomment if loading from Drive is very slow (large files)
# import shutil
# shutil.copy(f"{EMBED_DIR}/wav2vec2_embeddings.npy", "/content/wav2vec2_embeddings.npy")
# shutil.copy(f"{EMBED_DIR}/wav2vec2_seg_ids.npy",    "/content/wav2vec2_seg_ids.npy")
# X_wav2vec   = np.load("/content/wav2vec2_embeddings.npy")
# seg_ids_w2v = np.load("/content/wav2vec2_seg_ids.npy", allow_pickle=True)

# ── Load wav2vec2 embeddings ─────────────────────────────────────
print("Loading wav2vec2 embeddings...")
X_wav2vec   = np.load(f"{EMBED_DIR}/wav2vec2_embeddings.npy")
seg_ids_w2v = np.load(f"{EMBED_DIR}/wav2vec2_seg_ids.npy", allow_pickle=True)
print(f"  Embeddings shape: {X_wav2vec.shape}")

# ── Load prosodic features ────────────────────────────────────────
print("Loading prosodic features...")
prosodic_df  = pd.read_csv(f"{EMBED_DIR}/prosodic_features.csv")
prosodic_map = prosodic_df.set_index("dp_id")[
    ["f0_mean","f0_var","speaking_rate","pause_freq","pause_dur_mean","npvi"]
].to_dict("index")
print(f"  Prosodic utterances: {len(prosodic_map)}")

# ── Load segment manifest ─────────────────────────────────────────
seg_df = pd.read_csv(MANIFEST_IN)
print(f"  Total segments in manifest: {len(seg_df)}")

seg_id_to_idx = {sid: i for i, sid in enumerate(seg_ids_w2v)}

fused_features   = []
labels           = []
parent_ids       = []
seg_ids_final    = []
aug_types_list   = []
seg_indices_list = []
missing_w2v      = 0
missing_pros     = 0

for _, row in seg_df.iterrows():
    seg_id = row["seg_id"]

    if seg_id not in seg_id_to_idx:
        missing_w2v += 1
        continue

    w2v_embed   = X_wav2vec[seg_id_to_idx[seg_id]]   # (768,)
    parent_id   = row["parent_id"]
    base_parent = parent_id.split("_aug")[0] if "_aug" in parent_id else parent_id

    if base_parent in prosodic_map:
        pros_vec = np.array(list(prosodic_map[base_parent].values()), dtype=np.float32)
    else:
        pros_vec = np.zeros(6, dtype=np.float32)
        missing_pros += 1

    fused_features.append(np.concatenate([w2v_embed, pros_vec]))  # (774,)
    labels.append(row["label"])
    parent_ids.append(parent_id)
    seg_ids_final.append(seg_id)
    aug_types_list.append(row.get("aug_type", "original"))
    seg_indices_list.append(int(row.get("seg_index", 0)))

X      = np.array(fused_features)
y      = np.array(labels)
groups = np.array(parent_ids)
aug_types_arr   = np.array(aug_types_list)
seg_indices_arr = np.array(seg_indices_list, dtype=np.int32)

print(f"\nFused feature matrix shape : {X.shape}")
print(f"Label distribution — Native: {(y==1).sum()}, Non-Native: {(y==0).sum()}")
if missing_w2v:
    print(f"WARNING: {missing_w2v} segments skipped (missing wav2vec2 embedding)")
if missing_pros:
    print(f"WARNING: {missing_pros} segments used zero prosodic vector")

# ── Group-level Train/Test Split ──────────────────────────────────
# Split by parent_id so no recording appears in both train and test
gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f"\nTrain segments : {X_train.shape[0]}")
print(f"Test segments  : {X_test.shape[0]}")
print(f"Train — Native: {(y_train==1).sum()}, Non-Native: {(y_train==0).sum()}")
print(f"Test  — Native: {(y_test==1).sum()},  Non-Native: {(y_test==0).sum()}")

# ── Normalize — fit on train only ────────────────────────────────
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)    # transform only — no refit

# ── Save ──────────────────────────────────────────────────────────
np.savez(
    FEATURES_OUT,
    X_train=X_train_sc,
    X_test=X_test_sc,
    y_train=y_train,
    y_test=y_test,
    train_idx=train_idx,
    test_idx=test_idx,
    train_parent_ids=groups[train_idx],
    test_parent_ids=groups[test_idx],
    train_aug_types=aug_types_arr[train_idx],
    test_aug_types=aug_types_arr[test_idx],
    train_seg_indices=seg_indices_arr[train_idx],
    test_seg_indices=seg_indices_arr[test_idx],
)
joblib.dump(scaler, f"{PROJECT_ROOT}/outputs/scaler.joblib")

print(f"\nFeatures saved : {FEATURES_OUT}")
print(f"Scaler saved   : {PROJECT_ROOT}/outputs/scaler.joblib")

Loading wav2vec2 embeddings...
  Embeddings shape: (29222, 1024)
Loading prosodic features...
  Prosodic utterances: 234
  Total segments in manifest: 29222

Fused feature matrix shape : (29222, 1030)
Label distribution — Native: 9572, Non-Native: 19650

Train segments : 25854
Test segments  : 3368
Train — Native: 7650, Non-Native: 18204
Test  — Native: 1922,  Non-Native: 1446

Features saved : /content/drive/MyDrive/team_databaes/outputs/stage7_features.npz
Scaler saved   : /content/drive/MyDrive/team_databaes/outputs/scaler.joblib


In [13]:
# ============================================================
# STAGE 8: CLASSIFICATION
# Primary: LSTM feature extractor + SVM RBF classifier
# Baseline: Logistic Regression
#
# Architecture:
#   1. Group segment-level features into recording-level
#      sequences (keyed by parent_id + aug_type, sorted
#      by seg_index so temporal order is preserved).
#   2. Bidirectional LSTM encodes each variable-length
#      sequence → fixed 128-dim hidden-state vector.
#   3. SVM with RBF kernel classifies the LSTM features.
#
# WHY SVM RBF instead of RF?
#   SVM with RBF often generalises better than RF on small
#   datasets with high-dimensional feature spaces. The LSTM
#   output is only 128-dim — compact enough that RBF kernel
#   tricks work well without O(n^2) blowup on a huge dataset.
#   On small-to-medium N (<10k recordings) RBF SVM is fast
#   and typically achieves better margin generalisation.
#
# COLAB NOTES:
#   - LSTM training benefits from GPU (Runtime → GPU)
#   - SVM runs on CPU (sklearn)
#   - Expect ~5-20 min total depending on dataset size
#   - Models saved to Drive immediately after training
# ============================================================
import os
import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold, GroupShuffleSplit
from sklearn.metrics import f1_score, classification_report

# ── Google Drive path config ─────────────────────────────────────
PROJECT_ROOT = "/content/drive/MyDrive/team_databaes"

FEATURES_IN = f"{PROJECT_ROOT}/outputs/stage7_features.npz"
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(f"{PROJECT_ROOT}/outputs", exist_ok=True)

# ── Load features ─────────────────────────────────────────────────
print("Loading features from Drive...")
data    = np.load(FEATURES_IN, allow_pickle=True)
X_train = data["X_train"]
X_test  = data["X_test"]
y_train = data["y_train"]
y_test  = data["y_test"]

train_parent_ids  = data["train_parent_ids"]
test_parent_ids   = data["test_parent_ids"]
train_aug_types   = data["train_aug_types"]
test_aug_types    = data["test_aug_types"]
train_seg_indices = data["train_seg_indices"]
test_seg_indices  = data["test_seg_indices"]

INPUT_DIM = X_train.shape[1]   # 774

print(f"Training on {X_train.shape[0]} segments, {INPUT_DIM} features")
print(f"Test set: {X_test.shape[0]} segments")
print(f"Train label distribution — Native: {(y_train==1).sum()}, Non-Native: {(y_train==0).sum()}")
print(f"Using device: {DEVICE}")

# ── SKIP IF ALREADY TRAINED ───────────────────────────────────────
_lstm_path = f"{PROJECT_ROOT}/outputs/lstm_encoder.pt"
_svm_path  = f"{PROJECT_ROOT}/outputs/svm_model.joblib"
if os.path.exists(_lstm_path) and os.path.exists(_svm_path):
    print(f"\n⏭️  Models already exist — skipping training.")
    lstm_config = joblib.load(f"{PROJECT_ROOT}/outputs/lstm_config.joblib")
    test_data   = np.load(f"{PROJECT_ROOT}/outputs/stage8_test_data.npz", allow_pickle=True)
    svm = joblib.load(_svm_path)
    y_pred = svm.predict(test_data["X_test_lstm"])
    y_true = test_data["y_test_seq"]
    print(f"Test F1  : {f1_score(y_true, y_pred):.4f}")
    print("\nClassification report (test set — recording level):")
    print(classification_report(y_true, y_pred, target_names=["Non-Native", "Native"]))
    print("   Delete outputs/lstm_encoder.pt and outputs/svm_model.joblib to force re-training.")
    print("\n✅ Stage 8 already done — skipped.")
    raise SystemExit(0)

# ══════════════════════════════════════════════════════════════════
# GROUP SEGMENTS INTO RECORDING-LEVEL SEQUENCES
# ══════════════════════════════════════════════════════════════════
def group_into_sequences(X, y, parent_ids, aug_types, seg_indices):
    """
    Group segment features into recording-level sequences.
    Key = (parent_id, aug_type) so each augmentation variant
    forms its own temporal sequence sorted by seg_index.
    """
    seq_keys    = np.array([f"{pid}__{aug}" for pid, aug
                            in zip(parent_ids, aug_types)])
    unique_keys = np.unique(seq_keys)

    sequences    = []
    seq_labels   = []
    seq_lengths  = []
    seq_key_list = []

    for key in unique_keys:
        mask       = seq_keys == key
        indices    = seg_indices[mask]
        sort_order = np.argsort(indices)

        seq   = X[mask][sort_order]        # (n_segs, 774)
        label = int(y[mask][0])

        sequences.append(torch.FloatTensor(seq))
        seq_labels.append(label)
        seq_lengths.append(len(seq))
        seq_key_list.append(key)

    return sequences, np.array(seq_labels), seq_lengths, seq_key_list


train_seqs, y_train_seq, train_lens, train_keys = group_into_sequences(
    X_train, y_train, train_parent_ids, train_aug_types, train_seg_indices
)
test_seqs, y_test_seq, test_lens, test_keys = group_into_sequences(
    X_test, y_test, test_parent_ids, test_aug_types, test_seg_indices
)

print(f"\nRecording-level sequences:")
print(f"  Train: {len(train_seqs)} sequences "
      f"(from {len(np.unique(train_parent_ids))} recordings)")
print(f"  Test : {len(test_seqs)} sequences "
      f"(from {len(np.unique(test_parent_ids))} recordings)")
print(f"  Train — Native: {(y_train_seq==1).sum()}, "
      f"Non-Native: {(y_train_seq==0).sum()}")
print(f"  Test  — Native: {(y_test_seq==1).sum()},  "
      f"Non-Native: {(y_test_seq==0).sum()}")
print(f"  Avg sequence length — Train: {np.mean(train_lens):.1f}, "
      f"Test: {np.mean(test_lens):.1f}")

# ══════════════════════════════════════════════════════════════════
# TRAIN / VALIDATION SPLIT (by parent_id to prevent leakage)
# ══════════════════════════════════════════════════════════════════
train_parent_ids_seq = np.array([k.split("__")[0] for k in train_keys])

gss_val = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
tr_idx, val_idx = next(gss_val.split(
    np.arange(len(train_seqs)), y_train_seq, groups=train_parent_ids_seq
))

val_seqs    = [train_seqs[i] for i in val_idx]
val_labels  = y_train_seq[val_idx]
val_lens    = [train_lens[i] for i in val_idx]

actual_train_seqs   = [train_seqs[i] for i in tr_idx]
actual_train_labels = y_train_seq[tr_idx]
actual_train_lens   = [train_lens[i] for i in tr_idx]

print(f"\nAfter validation split:")
print(f"  Actual train : {len(actual_train_seqs)} sequences")
print(f"  Validation   : {len(val_seqs)} sequences")

# ══════════════════════════════════════════════════════════════════
# LSTM ENCODER
# ══════════════════════════════════════════════════════════════════
HIDDEN_DIM    = 64         # reduced from 128 to prevent overfitting
NUM_LAYERS    = 1          # single layer — avoids overfitting on small data
DROPOUT       = 0.5        # strong regularization
LSTM_FEAT_DIM = HIDDEN_DIM * 2   # bidirectional → 128


class SequenceDataset(Dataset):
    def __init__(self, sequences, labels, lengths):
        self.sequences = sequences
        self.labels    = torch.FloatTensor(labels)
        self.lengths   = lengths

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx], self.lengths[idx]


def collate_fn(batch):
    seqs, labels, lengths = zip(*batch)
    padded = pad_sequence(seqs, batch_first=True)      # (B, max_len, 774)
    return padded, torch.stack(labels), torch.LongTensor(lengths)


class LSTMEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(
            input_dim, hidden_dim, num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=True,
        )
        self.drop = nn.Dropout(dropout)
        # Classification head — used during LSTM pre-training only
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def encode(self, x, lengths):
        """Return the fixed-size hidden-state vector (hidden_dim*2)."""
        packed = pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, (h_n, _) = self.lstm(packed)
        # Concat final forward + backward hidden states
        h_final = torch.cat([h_n[-2], h_n[-1]], dim=1)   # (B, hidden*2)
        h_final = self.drop(h_final)
        return h_final

    def forward(self, x, lengths):
        h = self.encode(x, lengths)
        logits = self.classifier(h).squeeze(-1)
        return logits, h


# ── Build model ───────────────────────────────────────────────────
model     = LSTMEncoder(INPUT_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

# Class-balanced loss (computed on actual training sequences after grouping)
n_pos = int((actual_train_labels == 1).sum())
n_neg = int((actual_train_labels == 0).sum())
print(f"  LSTM training class counts — Native (pos): {n_pos}, Non-Native (neg): {n_neg}")
print(f"  pos_weight = {n_neg}/{n_pos} = {n_neg / max(n_pos, 1):.3f}")
pos_weight = torch.tensor([n_neg / max(n_pos, 1)]).to(DEVICE)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

train_dataset = SequenceDataset(actual_train_seqs, actual_train_labels, actual_train_lens)
train_loader  = DataLoader(
    train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn
)

val_dataset = SequenceDataset(val_seqs, val_labels, val_lens)
val_loader  = DataLoader(
    val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn
)

# ── Train LSTM (monitoring VALIDATION loss for early stopping) ────
EPOCHS   = 100
PATIENCE = 15

best_val_loss    = float("inf")
patience_counter = 0

print(f"\nTraining LSTM encoder ({EPOCHS} epochs max, patience={PATIENCE})...")
print(f"  hidden_dim={HIDDEN_DIM}, num_layers={NUM_LAYERS}, dropout={DROPOUT}, lr=3e-4")

for epoch in range(EPOCHS):
    # ── Training phase ──
    model.train()
    total_train_loss = 0
    n_train_batches  = 0

    for batch_x, batch_y, batch_lens in train_loader:
        batch_x    = batch_x.to(DEVICE)
        batch_y    = batch_y.to(DEVICE)
        batch_lens = batch_lens.to(DEVICE)

        logits, _ = model(batch_x, batch_lens)
        loss = criterion(logits, batch_y)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_train_loss += loss.item()
        n_train_batches  += 1

    avg_train_loss = total_train_loss / n_train_batches

    # ── Validation phase ──
    model.eval()
    total_val_loss = 0
    n_val_batches  = 0

    with torch.no_grad():
        for batch_x, batch_y, batch_lens in val_loader:
            batch_x    = batch_x.to(DEVICE)
            batch_y    = batch_y.to(DEVICE)
            batch_lens = batch_lens.to(DEVICE)

            logits, _ = model(batch_x, batch_lens)
            loss = criterion(logits, batch_y)

            total_val_loss += loss.item()
            n_val_batches  += 1

    avg_val_loss = total_val_loss / max(n_val_batches, 1)
    scheduler.step(avg_val_loss)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"  Epoch {epoch+1:>3}/{EPOCHS}: "
              f"train_loss={avg_train_loss:.4f}  val_loss={avg_val_loss:.4f}")

    # ── Early stopping on VALIDATION loss ──
    if avg_val_loss < best_val_loss - 1e-4:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), _lstm_path)
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"  Early stopping at epoch {epoch+1} "
                  f"(best val_loss={best_val_loss:.4f})")
            break

# Reload best weights
model.load_state_dict(torch.load(_lstm_path, map_location=DEVICE))
print(f"Best LSTM validation loss: {best_val_loss:.4f}")

# ── Extract LSTM features ─────────────────────────────────────────
def extract_lstm_features(mdl, sequences, lengths, device):
    mdl.eval()
    features = []
    with torch.no_grad():
        for seq, length in zip(sequences, lengths):
            x = seq.unsqueeze(0).to(device)          # (1, seq_len, 774)
            l = torch.LongTensor([length]).to(device)
            h = mdl.encode(x, l)                     # (1, hidden*2)
            features.append(h.cpu().numpy().squeeze(0))
    return np.array(features)

print("\nExtracting LSTM features...")
X_train_lstm = extract_lstm_features(model, train_seqs, train_lens, DEVICE)
X_test_lstm  = extract_lstm_features(model, test_seqs,  test_lens,  DEVICE)
print(f"  Train LSTM features: {X_train_lstm.shape}")
print(f"  Test  LSTM features: {X_test_lstm.shape}")

# ══════════════════════════════════════════════════════════════════
# SVM RBF CLASSIFIER
# RBF kernel on the 128-dim LSTM features.
# GridSearchCV with StratifiedGroupKFold — keeps augmented variants
# of the same recording in the same fold (prevents data leakage).
# ══════════════════════════════════════════════════════════════════
print("\n" + "="*55)
print("  PRIMARY — SVM RBF on LSTM features")
print("="*55)

param_grid = {
    "C":     [0.1, 1, 10, 100],
    "gamma": ["scale", 0.1, 0.01, 0.001],
}

svm_rbf = SVC(
    kernel="rbf",
    class_weight="balanced",
    probability=True,       # Platt scaling → predict_proba for AUC/confidence
    random_state=42,
)

# StratifiedGroupKFold ensures augmented variants of the same recording
# stay in the same fold — prevents mild data leakage during CV.
train_groups_seq = np.array([k.split("__")[0] for k in train_keys])

cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)

search = GridSearchCV(
    svm_rbf,
    param_grid=param_grid,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
    verbose=2,
    refit=True,
)

print(f"Running GridSearchCV ({len(param_grid['C'])*len(param_grid['gamma'])} combos x 3-fold)...")
print(f"Estimated time: 1-5 min\n")
search.fit(X_train_lstm, y_train_seq, groups=train_groups_seq)

best_svm   = search.best_estimator_
best_C     = search.best_params_["C"]
best_gamma = search.best_params_["gamma"]
best_cv_f1 = search.best_score_

print(f"\nBest C      : {best_C}")
print(f"Best gamma  : {best_gamma}")
print(f"Best CV F1  : {best_cv_f1:.4f}")

# ── Evaluate ──────────────────────────────────────────────────────
y_pred_train = best_svm.predict(X_train_lstm)
y_pred_test  = best_svm.predict(X_test_lstm)

train_f1 = f1_score(y_train_seq, y_pred_train)
test_f1  = f1_score(y_test_seq,  y_pred_test)

print(f"\nTrain F1 : {train_f1:.4f}")
print(f"Test F1  : {test_f1:.4f}")

print("\nClassification report (test set — recording level):")
print(classification_report(y_test_seq, y_pred_test,
                            target_names=["Non-Native", "Native"]))

# ── Baseline: Logistic Regression ─────────────────────────────────
print("="*55)
print("  BASELINE — Logistic Regression on LSTM features")
print("="*55)

lr = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
lr.fit(X_train_lstm, y_train_seq)
lr_f1 = f1_score(y_test_seq, lr.predict(X_test_lstm))
print(f"Logistic Regression Test F1 : {lr_f1:.4f}")
print(f"SVM RBF improvement over LR : +{(test_f1 - lr_f1):.4f}")

# ── Save CV results ───────────────────────────────────────────────
cv_results = pd.DataFrame(search.cv_results_)
cv_results.to_csv(f"{PROJECT_ROOT}/outputs/stage8_cv_results.csv", index=False)

# ── Save everything ───────────────────────────────────────────────
lstm_config = {
    "input_dim":  INPUT_DIM,
    "hidden_dim": HIDDEN_DIM,
    "num_layers": NUM_LAYERS,
    "dropout":    DROPOUT,
}
joblib.dump(lstm_config, f"{PROJECT_ROOT}/outputs/lstm_config.joblib")
joblib.dump(best_svm,    _svm_path)
joblib.dump(lr,          f"{PROJECT_ROOT}/outputs/lr_baseline.joblib")
joblib.dump(search,      f"{PROJECT_ROOT}/outputs/grid_search_results.joblib")

# Recording-level test data (for Stage 9 evaluation)
np.savez(
    f"{PROJECT_ROOT}/outputs/stage8_test_data.npz",
    X_test_lstm=X_test_lstm,
    y_test_seq=y_test_seq,
    test_keys=np.array(test_keys),
)

print(f"\n✅ Stage 8 complete.")
print(f"   Primary model : LSTM encoder → SVM RBF (C={best_C}, gamma={best_gamma})")
print(f"\nModels saved to Drive:")
print(f"  {_lstm_path}")
print(f"  {_svm_path}")
print(f"  {PROJECT_ROOT}/outputs/lstm_config.joblib")
print(f"  {PROJECT_ROOT}/outputs/lr_baseline.joblib")
print(f"  {PROJECT_ROOT}/outputs/stage8_test_data.npz")

Loading features from Drive...
Training on 25854 segments, 1030 features
Test set: 3368 segments
Train label distribution — Native: 7650, Non-Native: 18204
Using device: cuda

Recording-level sequences:
  Train: 632 sequences (from 198 recordings)
  Test : 92 sequences (from 36 recordings)
  Train — Native: 278, Non-Native: 354
  Test  — Native: 62,  Non-Native: 30
  Avg sequence length — Train: 40.9, Test: 36.6

After validation split:
  Actual train : 536 sequences
  Validation   : 96 sequences
  LSTM training class counts — Native (pos): 236, Non-Native (neg): 300
  pos_weight = 300/236 = 1.271

Training LSTM encoder (100 epochs max, patience=15)...
  hidden_dim=64, num_layers=1, dropout=0.5, lr=3e-4
  Epoch   1/100: train_loss=0.7669  val_loss=0.7479
  Epoch   5/100: train_loss=0.5593  val_loss=0.6343
  Epoch  10/100: train_loss=0.2063  val_loss=0.5500
  Epoch  15/100: train_loss=0.0575  val_loss=0.6236
  Epoch  20/100: train_loss=0.0291  val_loss=0.6812
  Early stopping at epoch 2

In [14]:
# ============================================================
# STAGE 9: EVALUATION
# Metrics: Accuracy, F1 (primary), Precision, Recall,
#          ROC AUC, EER, Confusion Matrix, 95% CI
#
# Evaluation is at RECORDING level — one prediction per
# recording sequence produced by LSTM encoder → SVM RBF.
#
# COLAB NOTES:
#   - Pure CPU — no GPU needed, runs in seconds
#   - Results saved to Drive as stage9_metrics.csv
# ============================================================
import os
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import (f1_score, confusion_matrix, accuracy_score,
                              roc_curve, precision_score, recall_score,
                              roc_auc_score)
from scipy import stats

# ── Google Drive path config ─────────────────────────────────────
PROJECT_ROOT = "/content/drive/MyDrive/team_databaes"

os.makedirs(f"{PROJECT_ROOT}/outputs", exist_ok=True)

# ── Load recording-level test data and SVM model ──────────────────
print("Loading model and test data...")
test_data   = np.load(f"{PROJECT_ROOT}/outputs/stage8_test_data.npz",
                       allow_pickle=True)
X_test_lstm = test_data["X_test_lstm"]
y_test      = test_data["y_test_seq"]
test_keys   = test_data["test_keys"]

best_model = joblib.load(f"{PROJECT_ROOT}/outputs/svm_model.joblib")

y_pred   = best_model.predict(X_test_lstm)
y_scores = best_model.predict_proba(X_test_lstm)[:, 1]   # P(Native)

# ── Metrics ───────────────────────────────────────────────────────
acc       = accuracy_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall    = recall_score(y_test, y_pred, zero_division=0)
auc       = roc_auc_score(y_test, y_scores)
cm        = confusion_matrix(y_test, y_pred)

# EER: threshold where FPR == FRR
fpr, tpr, thresholds = roc_curve(y_test, y_scores)
frr     = 1 - tpr
eer_idx = np.argmin(np.abs(fpr - frr))
eer     = (fpr[eer_idx] + frr[eer_idx]) / 2
eer_thr = thresholds[eer_idx]

# 95% Binomial CI on accuracy
n = len(y_test)
ci_low, ci_high = stats.binom.interval(0.95, n, acc)
ci_low  /= n
ci_high /= n

# ── Print report ──────────────────────────────────────────────────
sep = "=" * 55
print(f"\n{sep}")
print("  EVALUATION REPORT — Stage 9  (LSTM + SVM RBF)")
print(sep)
print(f"\n  Test set size : {n} recording sequences")
print(f"\n  Accuracy  : {acc*100:.2f}%")
print(f"  F1 Score  : {f1:.4f}  ← primary metric")
print(f"  Precision : {precision:.4f}")
print(f"  Recall    : {recall:.4f}")
print(f"  ROC AUC   : {auc:.4f}")
print(f"  EER       : {eer*100:.2f}%  (threshold={eer_thr:.4f})")
print(f"\n  95% CI on Accuracy: [{ci_low*100:.1f}%, {ci_high*100:.1f}%]")
print(f"\n  Confusion Matrix:")
print(f"  (Rows = True class, Cols = Predicted class)")
print(f"              Pred Non-Native  Pred Native")
print(f"  True Non-Nat   {cm[0,0]:>6}          {cm[0,1]:>6}")
print(f"  True Native    {cm[1,0]:>6}          {cm[1,1]:>6}")
print(f"\n  False Positives (non-native → native) : {cm[0,1]}")
print(f"  False Negatives (native → non-native) : {cm[1,0]}")
print(sep)

if eer < 0.05:
    print("  EER < 5%   → Excellent, production-grade")
elif eer < 0.10:
    print("  EER 5–10%  → Good, solid research-grade")
elif eer < 0.20:
    print("  EER 10–20% → Acceptable")
else:
    print("  EER > 20%  → Needs improvement")
print(sep)

# ── Dialect breakdown ─────────────────────────────────────────────
try:
    seg_df = pd.read_csv(f"{PROJECT_ROOT}/outputs/stage5_manifest.csv")
    parent_to_dialect = (seg_df.drop_duplicates("parent_id")
                         .set_index("parent_id")["dialect"].to_dict())

    test_parents  = [k.split("__")[0] for k in test_keys]
    test_dialects = [parent_to_dialect.get(p, "unknown") for p in test_parents]

    test_df = pd.DataFrame({
        "dialect": test_dialects,
        "y_true":  y_test,
        "y_pred":  y_pred,
        "correct": (y_pred == y_test),
    })

    print("\n  Accuracy by dialect (recording level):")
    for dialect, group in test_df.groupby("dialect"):
        acc_d = group["correct"].mean()
        n_d   = len(group)
        print(f"    {dialect:<20} {acc_d*100:.1f}%  (n={n_d})")
except Exception as e:
    print(f"\n  (Could not compute dialect breakdown: {e})")

# ── Save metrics ──────────────────────────────────────────────────
metrics = {
    "accuracy":      acc,
    "f1_score":      f1,
    "precision":     precision,
    "recall":        recall,
    "roc_auc":       auc,
    "eer":           eer,
    "eer_threshold": eer_thr,
    "ci_low":        ci_low,
    "ci_high":       ci_high,
    "n_test":        n,
    "tn": cm[0,0], "fp": cm[0,1],
    "fn": cm[1,0], "tp": cm[1,1],
}
pd.DataFrame([metrics]).to_csv(f"{PROJECT_ROOT}/outputs/stage9_metrics.csv", index=False)
print(f"\nSaved: {PROJECT_ROOT}/outputs/stage9_metrics.csv")

Loading model and test data...

  EVALUATION REPORT — Stage 9  (LSTM + SVM RBF)

  Test set size : 92 recording sequences

  Accuracy  : 84.78%
  F1 Score  : 0.8833  ← primary metric
  Precision : 0.9138
  Recall    : 0.8548
  ROC AUC   : 0.9237
  EER       : 17.20%  (threshold=0.4937)

  95% CI on Accuracy: [77.2%, 91.3%]

  Confusion Matrix:
  (Rows = True class, Cols = Predicted class)
              Pred Non-Native  Pred Native
  True Non-Nat       25               5
  True Native         9              53

  False Positives (non-native → native) : 5
  False Negatives (native → non-native) : 9
  EER 10–20% → Acceptable

  Accuracy by dialect (recording level):
    Arabic_AE            83.3%  (n=12)
    Arabic_MSA           100.0%  (n=22)
    Arabic_QA            73.1%  (n=26)
    Arabic_SA            80.8%  (n=26)
    Unknown_NN           100.0%  (n=6)

Saved: /content/drive/MyDrive/team_databaes/outputs/stage9_metrics.csv


In [15]:
# ============================================================
# STAGE 10: OUTPUT GENERATION
# Deliverables:
#   1. outputs/predictions.csv
#   2. outputs/technical_report.md
#
# COLAB NOTES:
#   - Runs in seconds — pure CPU, no heavy computation
#   - Both output files are saved directly to Drive
# ============================================================
import os
import numpy as np
import pandas as pd
import joblib
from datetime import date

# ── Google Drive path config ─────────────────────────────────────
PROJECT_ROOT = "/content/drive/MyDrive/team_databaes"

os.makedirs(f"{PROJECT_ROOT}/outputs", exist_ok=True)

# ── Load everything ───────────────────────────────────────────────
print("Loading models, features, and metrics...")
test_data    = np.load(f"{PROJECT_ROOT}/outputs/stage8_test_data.npz",
                        allow_pickle=True)
X_test_lstm  = test_data["X_test_lstm"]
y_test       = test_data["y_test_seq"]
test_keys    = test_data["test_keys"]

best_model = joblib.load(f"{PROJECT_ROOT}/outputs/svm_model.joblib")
metrics    = pd.read_csv(f"{PROJECT_ROOT}/outputs/stage9_metrics.csv").iloc[0]

# ── Predictions ───────────────────────────────────────────────────
y_pred          = best_model.predict(X_test_lstm)
probs           = best_model.predict_proba(X_test_lstm)
confidence      = probs.max(axis=1)
prob_native     = probs[:, 1]
prob_non_native = probs[:, 0]

# ── Map test_keys back to metadata ────────────────────────────────
try:
    seg_df = pd.read_csv(f"{PROJECT_ROOT}/outputs/stage5_manifest.csv")
    parent_to_dialect = (seg_df.drop_duplicates("parent_id")
                         .set_index("parent_id")["dialect"].to_dict())

    recording_ids = [k.split("__")[0] for k in test_keys]
    aug_types     = [k.split("__")[1] if "__" in k else "original"
                     for k in test_keys]
    dialects      = [parent_to_dialect.get(r, "unknown")
                     for r in recording_ids]
except Exception as e:
    print(f"Warning: could not load segment info ({e}). Using generic IDs.")
    recording_ids = [f"rec_{i:03d}" for i in range(len(y_test))]
    aug_types     = ["unknown"] * len(y_test)
    dialects      = ["unknown"] * len(y_test)

# ── predictions.csv ───────────────────────────────────────────────
results = pd.DataFrame({
    "recording_id":     recording_ids,
    "aug_type":         aug_types,
    "dialect":          dialects,
    "predicted_label":  y_pred,
    "predicted_class":  ["Native" if p == 1 else "Non-Native" for p in y_pred],
    "confidence_score": np.round(confidence, 4),
    "prob_native":      np.round(prob_native, 4),
    "prob_non_native":  np.round(prob_non_native, 4),
    "true_label":       y_test,
    "true_class":       ["Native" if t == 1 else "Non-Native" for t in y_test],
    "correct":          (y_pred == y_test),
})

predictions_path = f"{PROJECT_ROOT}/outputs/predictions.csv"
results.to_csv(predictions_path, index=False)
n_correct = results["correct"].sum()
n_total   = len(results)
print(f"predictions.csv saved ({n_total} rows, {n_correct}/{n_total} correct)")

# ── Load LSTM config and SVM best params for report ───────────────
try:
    lstm_config = joblib.load(f"{PROJECT_ROOT}/outputs/lstm_config.joblib")
    lstm_params_str = (
        f"input_dim={lstm_config['input_dim']}, "
        f"hidden_dim={lstm_config['hidden_dim']}, "
        f"num_layers={lstm_config['num_layers']}, "
        f"bidirectional=True"
    )
except Exception:
    lstm_params_str = "See stage8 logs"

try:
    grid_search = joblib.load(f"{PROJECT_ROOT}/outputs/grid_search_results.joblib")
    best_params = grid_search.best_params_
    best_cv_f1  = grid_search.best_score_
    svm_params_str = f"C={best_params.get('C','N/A')}, gamma={best_params.get('gamma','N/A')}"
except Exception:
    svm_params_str = "See stage8 logs"
    best_cv_f1     = float(metrics.get("f1_score", 0))

# ── technical_report.md ───────────────────────────────────────────
data    = np.load(f"{PROJECT_ROOT}/outputs/stage7_features.npz",
                   allow_pickle=True)
n_train = int(data["X_train"].shape[0])
n_test  = int(len(y_test))
today   = date.today().strftime("%B %d, %Y")

report = f"""# Technical Evaluation Report
## Team Databaes — Arabic Native/Non-Native Speech Classification
**Client:** Renan Partners Private Limited
**Date:** {today}

---

## 1. Executive Summary

A binary classifier was developed to distinguish Native from Non-Native Arabic speakers.
The model combines deep phonetic representations from a pretrained wav2vec2 model with
hand-crafted prosodic features. A bidirectional LSTM encodes recording-level temporal
patterns from variable-length segment sequences, and an SVM with RBF kernel produces
the final decision — chosen for its strong generalisation on small datasets with
compact, high-dimensional feature spaces.

**Final Results on Renan Test Set (n={n_test} recording sequences):**

| Metric | Value |
|--------|-------|
| Accuracy | {metrics['accuracy']*100:.2f}% |
| F1 Score | {metrics['f1_score']:.4f} |
| Precision | {metrics['precision']:.4f} |
| Recall | {metrics['recall']:.4f} |
| ROC AUC | {metrics['roc_auc']:.4f} |
| EER | {metrics['eer']*100:.2f}% |
| 95% CI (Accuracy) | [{metrics['ci_low']*100:.1f}%, {metrics['ci_high']*100:.1f}%] |

---

## 2. Methodology

### 2.1 Data
- **Training source:** 160 Renan recordings (Gulf dialects: SA/QA/AE/KW) +
  200 Mozilla Common Voice 24.0 Arabic clips (native reference)
- **Augmentation:** Time stretching (±10%), pitch shifting (±2 semitones), Gaussian noise —
  training segments only. Non-native (minority) received 5 augmentations; native received 1.

### 2.2 Feature Extraction

**Stream 1 — Phonetic Embeddings (768-dim)**
- Model: `jonatasgrosman/wav2vec2-large-xlsr-53-arabic` (Baevski et al., 2020)
- Used frozen — no fine-tuning (prevents overfitting on 160 training recordings)
- Each 3-second segment → mean-pooled hidden states → 768-dimensional vector

**Stream 2 — Prosodic Features (6-dim)**

| Feature | Description |
|---------|-------------|
| F0 mean | Average fundamental frequency |
| F0 variance | Pitch dynamics |
| Speaking rate | Syllable-proxy peaks per second |
| Pause frequency | Pauses (>200ms) per second |
| Mean pause duration | Average pause length in seconds |
| nPVI | Normalised Pairwise Variability Index (rhythmic stress-timing) |

### 2.3 Fusion and Normalisation
- Streams concatenated → 774-dimensional vector per segment
- StandardScaler fitted on training data only
- Segments grouped by recording into temporal sequences

### 2.4 Classifier

**Stage 1 — LSTM Temporal Encoder**
- Architecture: Bidirectional LSTM ({lstm_params_str})
- Processes variable-length sequences of 774-dim segment features per recording
- Outputs a fixed 128-dim recording-level representation
- Trained end-to-end with BCE loss (class-balanced via pos_weight) + early stopping

**Stage 2 — SVM with RBF Kernel**
- Best hyperparameters: {svm_params_str}
- Best 3-fold CV F1: {best_cv_f1:.4f}
- `class_weight='balanced'`, `probability=True` (Platt scaling for confidence scores)
- RBF kernel chosen over RF for stronger margin generalisation on the compact
  128-dim LSTM features with limited training recordings

---

## 3. Results

### 3.1 Confusion Matrix

|  | Predicted Non-Native | Predicted Native |
|--|--|--|
| **True Non-Native** | {int(metrics['tn'])} | {int(metrics['fp'])} |
| **True Native** | {int(metrics['fn'])} | {int(metrics['tp'])} |

- **False Positives** (non-native classified as native): {int(metrics['fp'])}
- **False Negatives** (native classified as non-native): {int(metrics['fn'])}

### 3.2 Statistical Caveat
The 95% CI [{metrics['ci_low']*100:.1f}%, {metrics['ci_high']*100:.1f}%] is relatively wide due to the
small test set (n={n_test}). Results would be more stable with n≥100 test samples.

---

## 4. Known Limitations

1. **Small test set** — wide confidence intervals
2. **Kuwaiti dialect** — only ~1.9% of training data (3 files); may generalise poorly to KW test samples
3. **Short Common Voice clips** — 3–10s clips yield less reliable prosodic estimates
4. **Non-native L1 diversity** — model trained primarily on Gulf-native speakers

---

## 5. References

- Baevski et al. (2020). wav2vec 2.0. https://arxiv.org/abs/2006.11477
- Hochreiter & Schmidhuber (1997). Long Short-Term Memory. Neural Computation, 9(8).
- Cortes & Vapnik (1995). Support-Vector Networks. Machine Learning, 20(3).
- Emara & Shaker (2024). Speech Communication, 157.
- Grabe & Low (2002). Durational variability and the Rhythm Class Hypothesis.
- Sharma et al. (2021). SVM-based speech classification.
- Mozilla Common Voice 24.0 Arabic, CC0-1.0.
- jonatasgrosman/wav2vec2-large-xlsr-53-arabic. https://huggingface.co/jonatasgrosman/wav2vec2-large-xlsr-53-arabic

---
*Report generated automatically by Stage 10 — Team Databaes*
"""

report_path = f"{PROJECT_ROOT}/outputs/technical_report.md"
with open(report_path, "w") as f:
    f.write(report)

print("technical_report.md saved")
print("\n" + "="*50)
print("  ALL STAGE 10 DELIVERABLES COMPLETE")
print("="*50)
print(f"  {predictions_path}")
print(f"  {report_path}")
print("="*50)

Loading models, features, and metrics...
predictions.csv saved (92 rows, 78/92 correct)
technical_report.md saved

  ALL STAGE 10 DELIVERABLES COMPLETE
  /content/drive/MyDrive/team_databaes/outputs/predictions.csv
  /content/drive/MyDrive/team_databaes/outputs/technical_report.md
